# PDDL Attack Path Evaluation
Evaluate generated PDDL attack paths: solvability, syntax, and semantic quality.

## 1. Environment Setup (imports, constnats, global variables)

In [ ]:
import sys
import os
import re
import json
import resource
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import re as _re

import time
from datetime import datetime
import platform

from cve2pddlap.core.data_loader import load_few_shot_pool
from cve2pddlap.evaluation import create_ff_checker, create_enhsp_checker
from cve2pddlap.evaluation.problem_pddl_generator import generate_problem
from cve2pddlap.llm_providers.remote.openai_compat import QwenProvider


import torch
from sentence_transformers import SentenceTransformer, util as st_util
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline as hf_pipeline

from sklearn.metrics import classification_report, precision_recall_curve
from sklearn.model_selection import GroupKFold

from openai import OpenAI

from jinja2 import Environment, FileSystemLoader
from ipywidgets import interact, IntSlider

from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
from statsmodels.stats.inter_rater import fleiss_kappa as _fleiss_kappa, aggregate_raters
from collections import defaultdict

import pandas as pd
from collections import defaultdict

# Project root (relative — works from notebooks/attack_paths/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

# Data paths and file names
DATASET_PATH = os.path.join(PROJECT_ROOT, 'resources', 'data', 'CVE-PDDL-NNL-ReAP')
BAD_PDDL_DIR = os.path.join(PROJECT_ROOT, 'resources', 'data', 'mutated_bad_PDDL_AP')
TARGET_POOL_FILE = os.path.join(PROJECT_ROOT, 'resources', 'data', 'target_pool.json')
GENERATED_DOMAIN_DIR = os.path.join(PROJECT_ROOT, 'generated_domain')
FIG_DIR = os.path.join(PROJECT_ROOT, 'Fig', 'embedding_intrinsic')
os.makedirs(FIG_DIR, exist_ok=True)
FIG_DIR_EXT = os.path.join(PROJECT_ROOT, 'Fig', 'embedding_extrinsic')
os.makedirs(FIG_DIR_EXT, exist_ok=True)
EVAL_SET_DIR = os.path.join(GENERATED_DOMAIN_DIR, 'eval_set')
PROMPTS_PATH = os.path.join(PROJECT_ROOT, 'resources', 'prompt', 'evaluation')
AP_PATTERN = re.compile(r'^AP\d+$')
DOMAIN_FILE = 'domain.pddl'
PROBLEM_FILE = 'problem.pddl'

# Models (small defaults — replace with preferred models)
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
# EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-8B"  # too large for CPU
# EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-4B"  # Qwen3 embedding model 4B
# EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-4B"  # 4B alternative
# EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"  # fallback: smaller model


LLM_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# Generation parameters
SEED = 42
TEMPERATURE = 0.0
TOP_K = 1


@dataclass(frozen=True)
class EvaluationFlags:
    syntax_check: bool = True
    embedding_intrinsic: bool = True
    embedding_extrinsic: bool = True
    llm_intrinsic: bool = True
    llm_extrinsic: bool = True


eval_flags = EvaluationFlags()

os.environ['MallocStackLogging'] = '0'

# Results output directory
RESULTS_BASE = os.path.join(PROJECT_ROOT, "results", "tests", "reference_set")

def get_device_info():
    """Return device info dict."""
    import torch
    return {
        "platform": platform.platform(),
        "processor": platform.processor(),
        "python": platform.python_version(),
        "torch_device": "cuda" if torch.cuda.is_available() else "cpu",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
    }
import matplotlib.pyplot as plt
from dotenv import load_dotenv


- Why `BAAI/bge-base-en-v1.5`?
  - Strong MTEB performance. On the [Massive Text Embedding Benchmark](https://huggingface.co/spaces/mteb/leaderboard), bge-base is among the top-performing models in its size class (approximately 110M parameters).
  - Cross-modal capability for code and natural language. The bge series performs well on tasks such as code search and code–natural language matching, making it suitable for measuring cross-modal similarity between natural-language CVE descriptions and PDDL code.

## 2. Load data, Prompts, LLM (tokenizer and embedding model)

In [133]:
few_shot_pool = load_few_shot_pool(DATASET_PATH)
print(f'Reference examples: {len(few_shot_pool)}')
for ex in few_shot_pool[:5]:
    print(f'  {ex.key}')
print('  ...')
print(f'Generated domain dir to be evaluated: {os.path.relpath(EVAL_SET_DIR)}')

Reference examples: 55
  CVE-2022-1471 / AP1
  CVE-2022-40149 / AP1
  CVE-2022-40149 / AP2
  CVE-2022-40150 / AP1
  CVE-2022-40150 / AP2
  ...
Generated domain dir to be evaluated: ../../generated_domain/eval_set


In [134]:
def model_short_name(model_name):
    """Generate a short readable name from a full model identifier.
    e.g. 'meta/llama-3.3-70b-instruct' -> 'llama-3.3-70b'
         'gpt-4.1-mini' -> 'gpt-4.1-mini'
         'all-MiniLM-L6-v2' -> 'all-MiniLM-L6-v2'
         'BAAI/bge-base-en-v1.5' -> 'bge-base-en-v1.5'
         'Qwen/Qwen3-Embedding-4B' -> 'Qwen3-Embedding-4B'
    """
    name = model_name.split("/")[-1]  # strip org prefix
    for suffix in ["-instruct", "-Instruct", "-chat", "-Chat"]:
        if name.endswith(suffix):
            name = name[:-len(suffix)]
            break
    return name


def load_target_pool(target_pool_file):
    """Load CVE descriptions from target_pool.json. Returns {cve_id: description}."""
    with open(target_pool_file, encoding='utf-8') as f:
        pool = json.load(f)
    return {entry['cve_id']: entry['description'] for entry in pool}


def load_dataset(data_path, cve_descriptions):
    """Load all CVEs with their descriptions (from target_pool) and attack path PDDL files."""
    dataset = []
    for cve_dir in sorted(Path(data_path).iterdir()):
        if not cve_dir.is_dir():
            continue
        cve_id = cve_dir.name
        description = cve_descriptions.get(cve_id)
        if description is None:
            continue
        attack_paths = []
        for ap_dir in sorted(cve_dir.iterdir()):
            if not ap_dir.is_dir() or not AP_PATTERN.match(ap_dir.name):
                continue
            domain_file = ap_dir / DOMAIN_FILE
            problem_file = ap_dir / PROBLEM_FILE
            if domain_file.exists() and problem_file.exists():
                attack_paths.append({
                    'ap_id': ap_dir.name,
                    'domain': domain_file.read_text(encoding='utf-8').strip(),
                    'problem': problem_file.read_text(encoding='utf-8').strip(),
                })
        dataset.append({
            'cve_id': cve_id,
            'description': description,
            'attack_paths': attack_paths,
        })
    return dataset


def load_bad_dataset(bad_pddl_dir, cve_descriptions):
    """Load mutated bad PDDL domains from mutated_bad_PDDL_AP/.
    Returns list of {cve_id, description, attack_paths: [{ap_id, domain}]}."""
    bad_dataset = []
    bad_dir = Path(bad_pddl_dir)
    if not bad_dir.exists():
        print(f"WARNING: {bad_pddl_dir} not found")
        return bad_dataset
    for cve_dir in sorted(bad_dir.iterdir()):
        if not cve_dir.is_dir():
            continue
        cve_id = cve_dir.name
        description = cve_descriptions.get(cve_id, "")
        attack_paths = []
        for ap_dir in sorted(cve_dir.iterdir()):
            if not ap_dir.is_dir():
                continue
            domain_file = ap_dir / DOMAIN_FILE
            if domain_file.exists():
                attack_paths.append({
                    'ap_id': ap_dir.name,
                    'domain': domain_file.read_text(encoding='utf-8').strip(),
                })
        if attack_paths:
            bad_dataset.append({
                'cve_id': cve_id,
                'description': description,
                'attack_paths': attack_paths,
            })
    return bad_dataset

def load_prompts(prompts_path):
    """Load Jinja2 evaluation prompt templates."""
    return Environment(loader=FileSystemLoader(prompts_path))


def load_embedding_model(model_name):
    """Load a SentenceTransformer bi-encoder model.
    Qwen3-Embedding models require trust_remote_code=True.
    """
    if "Qwen3-Embedding" in model_name:
        return SentenceTransformer(model_name, trust_remote_code=True)
    return SentenceTransformer(model_name)


def load_llm(model_name):
    """Load a HuggingFace causal LLM with its tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float16,
        device_map='auto',
    )
    gen = hf_pipeline(
        'text-generation',
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        do_sample=False,
    )
    return gen, tokenizer

TARGET_POOL_TI_FILE = os.path.join(PROJECT_ROOT, "resources", "data", "target_pool_ti.json")

def load_target_pool_ti(path):
    """Load TI-enriched target pool. Returns {cve_id: {field: value}}."""
    with open(path, encoding="utf-8") as f:
        pool = json.load(f)
    return {entry["cve_id"]: entry for entry in pool}

def strip_base_score(cvss_nl):
    """Remove Base Score from CVSS vector NL string."""
    if not cvss_nl:
        return ""
    return _re.sub(r"\s*Base Score:\s*[\d.]+\.?\s*", "", cvss_nl).strip()

def build_nl_ti_selected(entry_ti):
    """Build TI-selected NL: desc + capec + cvss_vector_nl (no base score)."""
    parts = [entry_ti.get("description", "")]
    capec = entry_ti.get("capec", "")
    if capec:
        parts.append(capec)
    cvss = strip_base_score(entry_ti.get("cvss_vector_nl", ""))
    if cvss:
        parts.append(cvss)
    return " ".join(parts)

cve_descriptions = load_target_pool(TARGET_POOL_FILE)
dataset = load_dataset(DATASET_PATH, cve_descriptions)
bad_dataset = load_bad_dataset(BAD_PDDL_DIR, cve_descriptions)
print(f"Bad (mutated) examples: {len(bad_dataset)} CVEs, {sum(len(e['attack_paths']) for e in bad_dataset)} domains")
prompt_env = load_prompts(PROMPTS_PATH)

ti_pool = load_target_pool_ti(TARGET_POOL_TI_FILE)
print(f"TI pool loaded: {len(ti_pool)} CVEs")

embedding_model = None
if eval_flags.embedding_intrinsic or eval_flags.embedding_extrinsic:
    embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)

llm, tokenizer = None, None
if eval_flags.llm_intrinsic or eval_flags.llm_extrinsic:
    llm, tokenizer = load_llm(LLM_MODEL_NAME)

# --- Calibration data for LLM-as-expert evaluation ---
CALIBRATION_DATA_PATH = os.path.join(PROMPTS_PATH, "data.jsonl")

def load_calibration_data(data_jsonl_path, eval_type="intrinsic", exclude_cve=None, n=4, seed=42):
    """Sample n calibration examples from data.jsonl.
    
    Constraints:
        - exclude_cve: the CVE being evaluated is excluded (prevent data leakage)
        - n >= 2: at least 1 good + 1 bad example guaranteed
        - balanced: samples from both calibration_good and calibration_bad pools
    
    Args:
        data_jsonl_path: path to data.jsonl
        eval_type: 'intrinsic' or 'extrinsic'
        exclude_cve: CVE ID to exclude
        n: total number of calibration examples (>= 2)
        seed: random seed for reproducibility
    """
    import random as _rng
    rng = _rng.Random(seed)
    
    with open(data_jsonl_path) as f:
        all_data = [json.loads(line) for line in f]
    
    pool = [d for d in all_data
            if d.get("eval_type") == eval_type
            and d.get("role", "").startswith("calibration")
            and d.get("cve_id") != exclude_cve]
    
    good = [d for d in pool if d.get("role") == "calibration_good"]
    bad = [d for d in pool if d.get("role") == "calibration_bad"]
    
    n = max(n, 2)  # enforce minimum 2
    n_good = max(1, n // 2)       # at least 1 good
    n_bad = max(1, n - n_good)    # at least 1 bad
    # Adjust if one pool is too small
    n_good = min(n_good, len(good))
    n_bad = min(n_bad, len(bad))
    
    selected_good = rng.sample(good, n_good) if good else []
    selected_bad = rng.sample(bad, n_bad) if bad else []
    
    return selected_good + selected_bad


# --- Rate limit retry wrapper ---
def api_call_with_retry(func, *args, max_retries=5, base_delay=5, **kwargs):
    """Call func with exponential backoff on rate limit errors."""
    for attempt in range(max_retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                delay = base_delay * (2 ** attempt)
                print(f"  [RATE LIMIT] retry {attempt+1}/{max_retries} in {delay}s...")
                time.sleep(delay)
            else:
                raise
    raise RuntimeError(f"Max retries ({max_retries}) exceeded")


# --- Classification report helpers (following Marco's pattern) ---
from sklearn.metrics import classification_report

def report_row(y_true, y_pred, **meta):
    """Flatten classification_report into a single dict row, with TPR/FPR."""
    rpt = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    row = dict(meta)
    for key, val in rpt.items():
        if isinstance(val, dict):
            for metric, v in val.items():
                row[f'{key}__{metric}'] = v
        else:
            row[key] = val
    row['tpr'] = rpt.get('1', {}).get('recall', float('nan'))
    row['fpr'] = 1.0 - rpt.get('0', {}).get('recall', float('nan'))
    return row

def print_report(y_true, y_pred, title=''):
    if title:
        print(f'\n{title}')
        print('─' * len(title))
    print(classification_report(y_true, y_pred, zero_division=0))


Bad (mutated) examples: 18 CVEs, 55 domains
TI pool loaded: 26 CVEs


Device set to use cpu


## 3. Evaluation

Select two example:
1. reference
2. bad one

In [ ]:
# ── Build test samples: 1 random reference + 1 random bad ──
BAD_PDDL_DIR = os.path.join(PROJECT_ROOT, "resources", "data", "mutated_bad_PDDL_AP")

rng_test = np.random.default_rng(SEED)

# Pick 1 random reference AP
ref_entry = rng_test.choice(dataset)
ref_ap = rng_test.choice(ref_entry["attack_paths"])
test_samples = [("reference", ref_entry["cve_id"], ref_ap["ap_id"], ref_ap["domain"])]

# Pick 1 random bad AP
bad_domains = []
for cve_dir in sorted(Path(BAD_PDDL_DIR).iterdir()):
    if not cve_dir.is_dir():
        continue
    for ap_dir in sorted(cve_dir.iterdir()):
        if not ap_dir.is_dir():
            continue
        domain_file = ap_dir / "domain.pddl"
        if domain_file.exists():
            bad_domains.append((cve_dir.name, ap_dir.name, domain_file.read_text(encoding="utf-8")))

bad_pick = bad_domains[rng_test.integers(len(bad_domains))]
test_samples.append(("bad", bad_pick[0], bad_pick[1], bad_pick[2]))

print(f"Test samples: {len(test_samples)}")
for source, cve, ap, dom in test_samples:
    print(f"  [{source}] {cve}/{ap}  ({len(dom)} chars)")

Test samples: 2
  [reference] CVE-2022-40149/AP2  (11294 chars)
  [bad] CVE-2024-38809/AP1_inject_capability_violation_rep1  (11579 chars)


Metric: domain statistics

In [23]:
def domain_stats(domain_pddl):
    """Extract structural stats from a PDDL domain string."""
    return {
        "domain_size_bytes": len(domain_pddl.encode("utf-8")),
        "n_actions": len(re.findall(r"\(:action\s", domain_pddl)),
        "n_predicates": len(re.findall(r"\([\w-]+", re.findall(r"\(:predicates([^)]*(?:\([^)]*\))*[^)]*?)\)", domain_pddl, re.DOTALL)[0])) if re.findall(r"\(:predicates", domain_pddl) else 0,
        "n_types": len(re.findall(r"\(:types([^)]*?)\)", domain_pddl, re.DOTALL)[0].split()) if re.findall(r"\(:types", domain_pddl) else 0,
    }

Metric: peak memory in MB (gpu/cpu)

In [24]:
def get_peak_rss_mb():
    """Get current process peak RSS in MB (macOS: bytes, Linux: KB)."""
    import platform
    ru = resource.getrusage(resource.RUSAGE_CHILDREN)
    if platform.system() == "Darwin":
        return round(ru.ru_maxrss / 1024 / 1024, 2)
    return round(ru.ru_maxrss / 1024, 2)


### 3.1 Syntax Check (ENHSP)

In [25]:
enhsp = create_enhsp_checker()
# ── Run syntax check ──
t_start_syntax = time.time()
reference_syntax_results = []

save_dir = os.path.join(RESULTS_BASE, "syntax")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, "results_syntax.jsonl")

open(results_path, "w").close()

for source, cve_id, ap_id, domain_pddl in test_samples:
    problem_str = generate_problem(domain_pddl)
    t0 = time.time()
    r_enhsp = enhsp.check_from_string(domain_pddl, problem_str)
    elapsed = time.time() - t0
    stats = domain_stats(domain_pddl)

    result = {"timestamp": datetime.now().isoformat(), 
        "source": source,
        "cve_id": cve_id,
        "ap_id": ap_id,
        "syntax_ok": r_enhsp.success,
        "error": r_enhsp.error,
        "elapsed_seconds": round(elapsed, 4),
        **stats,
    }
    reference_syntax_results.append(result)
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    print(f"[{source}] {cve_id}/{ap_id}  syntax: {r_enhsp.success}  {elapsed:.3f}s")

t_syntax = time.time() - t_start_syntax

n_pass = sum(1 for r in reference_syntax_results if r["syntax_ok"])
print(f"\nSyntax: {n_pass}/{len(reference_syntax_results)} passed, time: {t_syntax:.2f}s")


[reference] CVE-2022-40149/AP2  syntax: True  30.053s
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1  syntax: True  30.039s

Syntax: 2/2 passed, time: 60.12s


### 3.2 Solvability (Metric-FF)

In [27]:
ff = create_ff_checker()

# ── Run solvability check ──
t_start_solv = time.time()
reference_solvability_results = []

save_dir = os.path.join(RESULTS_BASE, "solvability")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, "results_solvability.jsonl")

open(results_path, "w").close()

for source, cve_id, ap_id, domain_pddl in test_samples:
    problem_str = generate_problem(domain_pddl)
    t0 = time.time()
    r_ff = ff.check_from_string(domain_pddl, problem_str)
    elapsed = time.time() - t0
    stats = domain_stats(domain_pddl)

    result = {"timestamp": datetime.now().isoformat(), 
        "source": source,
        "cve_id": cve_id,
        "ap_id": ap_id,
        "solvable": r_ff.solvable,
        "plan_length": r_ff.plan_length,
        "plan_cost": r_ff.plan_cost,
        "plan": r_ff.plan,
        "error": r_ff.error,
        "elapsed_seconds": round(elapsed, 4),
        **stats,
    }
    reference_solvability_results.append(result)
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    status = "SOLVABLE" if r_ff.solvable else "FAIL"
    print(f"[{source}] {cve_id}/{ap_id}  {status}  plan_length={r_ff.plan_length}  cost={r_ff.plan_cost}  {elapsed:.3f}s")
    if r_ff.plan:
        for i, action in enumerate(r_ff.plan):
            print(f"  {i}: {action}")

t_solv = time.time() - t_start_solv

n_solvable = sum(1 for r in reference_solvability_results if r["solvable"])
print(f"\nSolvability: {n_solvable}/{len(reference_solvability_results)} solvable, time: {t_solv:.2f}s")


[reference] CVE-2022-40149/AP2  SOLVABLE  plan_length=9  cost=38.0  0.036s
  0: ATTACKER-CRAFTS-MALICIOUS-XML-PAYLOAD SEFA XML-PAYLOAD_SEFA JAVA-LIBRARY_SEFA CVE_2022_40149
  1: ATTACKER-SENDS-HTTP-POST-REQUEST-WITH-MALICIOUS-PAYLOAD SEFA HTTP-POST-REQUEST_SEFA XML-PAYLOAD_SEFA API-ENDPOINT_SEFA CVE_2022_40149
  2: TARGET-SYSTEM-RECEIVES-HTTP-POST-REQUEST-WITH-MALICIOUS-PAYLOAD SEFA HTTP-POST-REQUEST_SEFA XML-PAYLOAD_SEFA API-ENDPOINT_SEFA
  3: TARGET-SYSTEM-EXTRACTS-XML-PAYLOAD-FROM-HTTP-REQUEST SEFA HTTP-POST-REQUEST_SEFA XML-PAYLOAD_SEFA SOAP-ENDPOINT_SEFA
  4: TARGET-SYSTEM-INVOKES-JETTISON-XML-TO-JSON-TRANSFORMATION SEFA JAVA-LIBRARY_SEFA XML-PAYLOAD_SEFA JSON-PAYLOAD_SEFA
  5: TARGET-SYSTEM-USES-JETTISON-JSON-PARSER SEFA JAVA-LIBRARY_SEFA JSON-PAYLOAD_SEFA
  6: TARGET-SYSTEM-JETTISON-PARSER-ALLOCATES-STACK-MEMORY-FOR-JSON-CONSTRUCTION JAVA-LIBRARY_SEFA JSON-PAYLOAD_SEFA SEFA
  7: TARGET-SYSTEM-TRIGGERS-STACKOVERFLOW-ERROR SEFA CVE_2022_40149
  8: TARGET-SYSTEM-BECOMES-UNRESPONSIV

### 3.3 Semantic Evaluation

#### 3.3.1 Intrinsic

##### 3.3.1.1a Embedding: NL CVE Description vs PDDL Similarity

Block 1:  the function to compute the embedding similarity between the NL CVE description vs the PDDL code (domain)

In [37]:
def embedding_similarity_intrinsic(descriptions, pddl_texts, model):
    """Cosine similarity matrix between NL descriptions and PDDL codes.
    Args:
        descriptions: list of CVE NL description strings
        pddl_texts: list of PDDL (domain) strings
        model: SentenceTransformer model
    Returns:
        numpy array of shape (len(descriptions), len(pddl_texts))
    """
    E_desc = model.encode(
        descriptions,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    E_pddl = model.encode(
        pddl_texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    S = E_desc @ E_pddl.T
    return S.float().cpu().numpy()

Block 2: Cross-validation, performance estimation, and corrected proportion

Process:
1. Build positive/negative pairs from reference data 
   - 1:PDDL doman vs same CVE NL desc 
   - 0: PDDL doman vs different CVE NL desc
2. CVE-level GroupKFold CV with bootstrap threshold on each train fold
3. Out-of-fold predictions → global confusion matrix → TPR, FPR, classification report
4. Apply calibrated threshold to generated domains → corrected proportion π = (PPV - FPR) / (TPR - FPR)

Why GroupKFold?
- one CVE NL desc corresponded to multiple PDDL AP domain 
- If the train fold contains (for example, CVE-2024-12798 NL desc vs domain_AP4) and the validation fold contains (CVE-2024-12798 NL desc vs domain_AP1), the threshold is calibrated on an embedding the validation set also shares, this is data leakage. GroupKFold ensures all pairs involving the same CVE NL desc are assigned to the same fold, eliminating this issue.

In [38]:
# Step 1: Build positive/negative pairs
def build_intrinsic_pairs(dataset, model):
    """
    Build (scores, labels, groups) for intrinsic embedding evaluation.
    Positive (1): CVE domain vs same CVE NL description
    Negative (0): CVE domain vs different CVE NL description
    Groups: assigned by the description-side CVE
    Reuses embedding_similarity_intrinsic for batch computation.
    """
    descs, domains, cve_ids = [], [], []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            descs.append(entry["description"])
            domains.append(ap["domain"])
            cve_ids.append(entry["cve_id"])

    sim_matrix = embedding_similarity_intrinsic(descs, domains, model)

    scores, labels, groups = [], [], []
    n = len(descs)
    for i in range(n):
        for j in range(n):
            scores.append(float(sim_matrix[i, j]))
            labels.append(1 if cve_ids[i] == cve_ids[j] else 0)
            groups.append(cve_ids[i])

    return np.array(scores), np.array(labels), np.array(groups)


pair_scores_a, pair_labels_a, pair_groups_a = build_intrinsic_pairs(dataset, embedding_model)
print(f"Embedding Model: {EMBEDDING_MODEL_NAME}")
print(f"  Positive pairs: {pair_labels_a.sum()}, Negative pairs: {(pair_labels_a == 0).sum()}")
print(f"  Groups (CVEs): {len(np.unique(pair_groups_a))}")

# Global list for cross-model comparison CSV
calibration_embedding_rows = []


Embedding Model: all-MiniLM-L6-v2
  Positive pairs: 203, Negative pairs: 2822
  Groups (CVEs): 21


In [39]:
# Step 2: CV calibration
def _find_threshold_pr(y_true, y_score):
    """Return the threshold closest to (1,1) in the precision-recall curve."""
    prec, rec, thr = precision_recall_curve(y_true, y_score)
    distances = np.sqrt((1 - prec[1:]) ** 2 + (1 - rec[1:]) ** 2)
    return float(thr[np.argmin(distances)])

def run_calibration(scores, labels, groups, k=5, n_bootstrap=500, random_state=42):
    """
    CVE-level GroupKFold CV with bootstrap threshold search on each train fold.
    Returns: median_threshold, fold_thresholds, y_pred (OOF), y_true (OOF)
    """
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    groups = np.asarray(groups)
    rng = np.random.RandomState(random_state)

    unique_groups = np.unique(groups)
    n_groups = len(unique_groups)
    actual_k = min(k, n_groups)
    if actual_k < k:
        print(f"Warning: only {n_groups} groups, reducing k from {k} to {actual_k}")

    gkf = GroupKFold(n_splits=actual_k)
    fold_thresholds, y_pred_parts, y_true_parts = [], [], []

    for fold_i, (train_idx, val_idx) in enumerate(gkf.split(scores, labels, groups)):
        tr_scores, tr_labels = scores[train_idx], labels[train_idx]
        bt = []
        for _ in range(n_bootstrap):
            idx = rng.choice(len(tr_scores), size=len(tr_scores), replace=True)
            bt.append(_find_threshold_pr(tr_labels[idx], tr_scores[idx]))
        fold_thr = float(np.median(bt))
        fold_thresholds.append(fold_thr)
        y_pred_parts.append((scores[val_idx] >= fold_thr).astype(int))
        y_true_parts.append(labels[val_idx])
        val_groups = np.unique(groups[val_idx])
        print(f"  Fold {fold_i+1}: threshold={fold_thr:.4f}, val CVEs={list(val_groups)}")

    return (
        float(np.median(fold_thresholds)),
        fold_thresholds,
        np.concatenate(y_pred_parts),
        np.concatenate(y_true_parts),
    )


cv_threshold_a, cv_fold_thr_a, cv_pred_a, cv_true_a = run_calibration(
    pair_scores_a, pair_labels_a, pair_groups_a)
print("Fold thresholds:", [f"{t:.4f}" for t in cv_fold_thr_a])
print(f"Median threshold: {cv_threshold_a:.4f}")


  Fold 1: threshold=0.3730, val CVEs=['CVE-2023-44487', 'CVE-2024-12798', 'CVE-2024-34447', 'CVE-2024-38809']
  Fold 2: threshold=0.4499, val CVEs=['CVE-2023-33202', 'CVE-2023-34055', 'CVE-2024-22243', 'CVE-2024-38816']
  Fold 3: threshold=0.4039, val CVEs=['CVE-2022-40150', 'CVE-2024-22262', 'CVE-2024-38820', 'CVE-2025-22228']
  Fold 4: threshold=0.4596, val CVEs=['CVE-2022-40149', 'CVE-2024-22259', 'CVE-2024-47072', 'CVE-2025-24813']
  Fold 5: threshold=0.4005, val CVEs=['CVE-2022-1471', 'CVE-2023-2976', 'CVE-2023-46589', 'CVE-2023-6378', 'CVE-2024-38286']
Fold thresholds: ['0.3730', '0.4499', '0.4039', '0.4596', '0.4005']
Median threshold: 0.4039


In [40]:
# Step 3: Performance report + save CV metrics

print(f"Embedding Model ({EMBEDDING_MODEL_NAME})")
print(classification_report(cv_true_a, cv_pred_a, zero_division=0))

row_a = report_row(cv_true_a, cv_pred_a,
                   metric="embedding", model=EMBEDDING_MODEL_NAME,
                   mode="intrinsic", threshold=cv_threshold_a)
print(f"TPR = {row_a['tpr']:.4f}  FPR = {row_a['fpr']:.4f}")

# ── Save CV calibration metrics (class 1 only) ──
save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")

cv_record = {
    "model": EMBEDDING_MODEL_NAME,
    "threshold": cv_threshold_a,
    "fold_thresholds": cv_fold_thr_a,
    "accuracy": row_a.get("accuracy", float("nan")),
    "precision": row_a.get("1__precision", float("nan")),
    "recall": row_a.get("1__recall", float("nan")),
    "f1": row_a.get("1__f1-score", float("nan")),
    "tpr": row_a["tpr"],
    "fpr": row_a["fpr"],
    "n_positive_pairs": int(pair_labels_a.sum()),
    "n_negative_pairs": int((pair_labels_a == 0).sum()),
}

# Deduplicate by model
existing = []
if os.path.exists(cv_path):
    with open(cv_path) as f:
        existing = [json.loads(line) for line in f if line.strip()]
    existing = [r for r in existing if r.get("model") != EMBEDDING_MODEL_NAME]
existing.append(cv_record)
with open(cv_path, "w") as f:
    for r in existing:
        f.write(json.dumps(r) + "\n")

# Append to global calibration rows
calibration_embedding_rows.append({
    "model": EMBEDDING_MODEL_NAME, "mode": "intrinsic", "threshold": cv_threshold_a,
    "precision": cv_record["precision"], "recall": cv_record["recall"],
    "f1": cv_record["f1"], "tpr": row_a["tpr"], "fpr": row_a["fpr"],
    "n_positive": int(pair_labels_a.sum()), "n_negative": int((pair_labels_a == 0).sum()),
})


Embedding Model (all-MiniLM-L6-v2)
              precision    recall  f1-score   support

           0       0.97      0.87      0.91      2822
           1       0.24      0.56      0.33       203

    accuracy                           0.85      3025
   macro avg       0.60      0.72      0.62      3025
weighted avg       0.92      0.85      0.88      3025

TPR = 0.5616  FPR = 0.1304


In [43]:
# Step 4: Apply threshold to test samples

def apply_threshold_single(description, domain_pddl, model, threshold):
    """Compute intrinsic similarity and apply threshold for a single domain."""
    E_desc = model.encode([description], convert_to_tensor=True, normalize_embeddings=True)
    E_pddl = model.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
    sim = float((E_desc * E_pddl).sum())
    pred = "True" if sim >= threshold else "False"
    return sim, pred

# def corrected_proportion(ppv, tpr, fpr):
#     """Corrected estimate of true positive proportion: pi = (PPV - FPR) / (TPR - FPR)"""
#     denom = tpr - fpr
#     if abs(denom) < 1e-10:
#         return float("nan")
#     return (ppv - fpr) / denom


# ── Apply to test_samples (1 reference + 1 bad) ──
save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, "results_intrinsic_similarity.jsonl")

open(results_path, "w").close()


print(f"Applying threshold {cv_threshold_a:.4f} ({EMBEDDING_MODEL_NAME}) to test samples:")
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    t0 = time.time()
    sim, pred = apply_threshold_single(description, domain_pddl, embedding_model, cv_threshold_a)
    elapsed = time.time() - t0
    result = {"timestamp": datetime.now().isoformat(), 
        "source": source,
        "model": EMBEDDING_MODEL_NAME,
        "threshold": cv_threshold_a,
        "cve_id": cve_id,
        "ap_id": ap_id,
        "similarity": round(sim, 6),
        "prediction": pred,
        "elapsed_seconds": round(elapsed, 4),
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    print(f"  [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred}  {elapsed:.3f}s")



# # ── Apply to ALL reference domains (uncomment for full run) ──
# def apply_threshold_batch(descriptions, domains, labels_list, model, threshold):
#     """Apply threshold using batch embedding_similarity_intrinsic. Returns results and PPV."""
#     sim_matrix = embedding_similarity_intrinsic(descriptions, domains, model)
#     results = []
#     for i, label in enumerate(labels_list):
#         sim = float(sim_matrix[i, i])
#         pred = "True" if sim >= threshold else "False"
#         results.append({"label": label, "similarity": sim, "prediction": pred})
#         print(f"  {label:45s} prediction: {pred} (similarity: {sim:.4f})")
#     predictions = [r["prediction"] for r in results]
#     ppv = sum(predictions) / len(predictions) if predictions else 0
#     return results, ppv
#
# ref_descs, ref_domains, ref_labels = [], [], []
# for entry in dataset:
#     for ap in entry["attack_paths"]:
#         ref_descs.append(entry["description"])
#         ref_domains.append(ap["domain"])
#         ref_labels.append(f"{entry['cve_id']}/{ap['ap_id']}")
#
# print(f"Applying threshold {cv_threshold_a:.4f} ({EMBEDDING_MODEL_NAME}) to reference domains:")
# ref_results_a, ref_ppv_a = apply_threshold_batch(ref_descs, ref_domains, ref_labels, embedding_model, cv_threshold_a)
# print(f"  Reference PPV: {ref_ppv_a:.4f} ({sum(r['prediction'] for r in ref_results_a)}/{len(ref_results_a)} predicted positive)")
#
# with open(results_path, "w") as f:
#     for r in ref_results_a:
#         r["model"] = EMBEDDING_MODEL_NAME
#         r["threshold"] = cv_threshold_a
#         f.write(json.dumps(r) + "\n")


Applying threshold 0.4039 (all-MiniLM-L6-v2) to test samples:
  [reference] CVE-2022-40149/AP2  sim=0.3739  pred=False  0.207s
  [bad] CVE-2024-38809/AP1_inject_capability_violation_rep1  sim=0.4631  pred=True  0.161s


In [119]:
# ── 3.3.1.1a with second embedding model ──
EMBEDDING_MODEL_NAME_2 = "BAAI/bge-base-en-v1.5"
embedding_model_2 = load_embedding_model(EMBEDDING_MODEL_NAME_2)
print(f"Loaded second embedding model: {EMBEDDING_MODEL_NAME_2}")

# Step 1: Build pairs
pair_scores_b, pair_labels_b, pair_groups_b = build_intrinsic_pairs(dataset, embedding_model_2)
print(f"Embedding Model: {EMBEDDING_MODEL_NAME_2}")
print(f"  Positive pairs: {pair_labels_b.sum()}, Negative pairs: {(pair_labels_b == 0).sum()}")

# Step 2: CV calibration
cv_threshold_b, cv_fold_thr_b, cv_pred_b, cv_true_b = run_calibration(
    pair_scores_b, pair_labels_b, pair_groups_b)
print(f"Median threshold: {cv_threshold_b:.4f}")

# Step 3: Performance report + save CV metrics
print(f"\nEmbedding Model ({EMBEDDING_MODEL_NAME_2})")
print(classification_report(cv_true_b, cv_pred_b, zero_division=0))

row_b = report_row(cv_true_b, cv_pred_b,
                   metric="embedding", model=EMBEDDING_MODEL_NAME_2,
                   mode="intrinsic", threshold=cv_threshold_b)
print(f"TPR = {row_b['tpr']:.4f}  FPR = {row_b['fpr']:.4f}")

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")

cv_record_b = {
    "model": EMBEDDING_MODEL_NAME_2,
    "threshold": cv_threshold_b,
    "fold_thresholds": cv_fold_thr_b,
    "accuracy": row_b.get("accuracy", float("nan")),
    "precision": row_b.get("1__precision", float("nan")),
    "recall": row_b.get("1__recall", float("nan")),
    "f1": row_b.get("1__f1-score", float("nan")),
    "tpr": row_b["tpr"],
    "fpr": row_b["fpr"],
    "n_positive_pairs": int(pair_labels_b.sum()),
    "n_negative_pairs": int((pair_labels_b == 0).sum()),
}

existing = []
if os.path.exists(cv_path):
    with open(cv_path) as f:
        existing = [json.loads(line) for line in f if line.strip()]
    existing = [r for r in existing if r.get("model") != EMBEDDING_MODEL_NAME_2]
existing.append(cv_record_b)
with open(cv_path, "w") as f:
    for r in existing:
        f.write(json.dumps(r) + "\n")

calibration_embedding_rows.append({
    "model": EMBEDDING_MODEL_NAME_2, "mode": "intrinsic", "threshold": cv_threshold_b,
    "precision": cv_record_b["precision"], "recall": cv_record_b["recall"],
    "f1": cv_record_b["f1"], "tpr": row_b["tpr"], "fpr": row_b["fpr"],
    "n_positive": int(pair_labels_b.sum()), "n_negative": int((pair_labels_b == 0).sum()),
})

# Step 4: Apply to test samples
results_path = os.path.join(save_dir, "results_intrinsic_similarity.jsonl")

# Append (don't clear — first model's results already in it)
print(f"\nApplying threshold {cv_threshold_b:.4f} ({EMBEDDING_MODEL_NAME_2}) to test samples:")
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    t0 = time.time()
    sim, pred = apply_threshold_single(description, domain_pddl, embedding_model_2, cv_threshold_b)
    elapsed = time.time() - t0
    pred_str = "True" if sim >= cv_threshold_b else "False"
    result = {"timestamp": datetime.now().isoformat(), "source": source, "model": EMBEDDING_MODEL_NAME_2,
              "threshold": cv_threshold_b, "cve_id": cve_id, "ap_id": ap_id,
              "similarity": round(sim, 6), "prediction": pred_str, "elapsed_seconds": round(elapsed, 4)}
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    print(f"  [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred_str}  {elapsed:.3f}s")


Loaded second embedding model: BAAI/bge-base-en-v1.5
Embedding Model: BAAI/bge-base-en-v1.5
  Positive pairs: 203, Negative pairs: 2822
  Fold 1: threshold=0.6850, val CVEs=['CVE-2023-44487', 'CVE-2024-12798', 'CVE-2024-34447', 'CVE-2024-38809']
  Fold 2: threshold=0.7337, val CVEs=['CVE-2023-33202', 'CVE-2023-34055', 'CVE-2024-22243', 'CVE-2024-38816']
  Fold 3: threshold=0.7105, val CVEs=['CVE-2022-40150', 'CVE-2024-22262', 'CVE-2024-38820', 'CVE-2025-22228']
  Fold 4: threshold=0.7337, val CVEs=['CVE-2022-40149', 'CVE-2024-22259', 'CVE-2024-47072', 'CVE-2025-24813']
  Fold 5: threshold=0.7337, val CVEs=['CVE-2022-1471', 'CVE-2023-2976', 'CVE-2023-46589', 'CVE-2023-6378', 'CVE-2024-38286']
Median threshold: 0.7337

Embedding Model (BAAI/bge-base-en-v1.5)
              precision    recall  f1-score   support

           0       0.96      0.95      0.96      2822
           1       0.41      0.46      0.44       203

    accuracy                           0.92      3025
   macro avg   

##### 3.3.1.2 LLM as an Expert: NL CVE Description vs PDDL Match

##### 3.3.1.2a Binary Intrinsic (True/False)
`llm_eval_intrinsic_binary`: binary classification using `completion.md.jinja` (binary=True, extrinsic=False)


In [86]:
# Unified evaluation template (binary/scored × intrinsic/extrinsic via parameters)
eval_template = prompt_env.get_template("completion.md.jinja")

def llm_eval_intrinsic_binary(cve_id, description, domain, client, model, seed=42, n_calibration=2):
    """Binary True/False classification: does the PDDL match the CVE?"""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "intrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=True, extrinsic=False,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description, domain=domain,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=512,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


 Block 2: call the function with the code and the specifications from the data set

In [18]:
# llm_intrinsic_results = []
# eval_set_path = Path(EVAL_SET_DIR)
# for cve_dir in sorted(eval_set_path.iterdir()):
#     if not cve_dir.is_dir():
#         continue
#     cve_id = cve_dir.name
#     description = cve_descriptions.get(cve_id)
#     if description is None:
#         print(f"SKIP {cve_id}: no description in target_pool.json")
#         continue
#     for config_dir in sorted(cve_dir.iterdir()):
#         if not config_dir.is_dir():
#             continue
#         domain_file = config_dir / DOMAIN_FILE
#         problem_file = config_dir / PROBLEM_FILE
#         if not domain_file.exists():
#             continue
#         domain = domain_file.read_text(encoding='utf-8').strip()
#         problem = problem_file.read_text(encoding='utf-8').strip() if problem_file.exists() else ''
#         response = llm_eval_intrinsic(description, domain, problem, llm)
#         llm_intrinsic_results.append({
#             'cve_id': cve_id,
#             'config': config_dir.name,
#             'llm_response': response,
#         })
#         print(f"{cve_id}/{config_dir.name}  response: {response}")

In [87]:
def preview_intrinsic_binary(n_calibration=2):
    sample = few_shot_pool[0]
    cal_data = load_calibration_data(CALIBRATION_DATA_PATH, 'intrinsic', exclude_cve=sample.cve_id)[:n_calibration]
    render_args = dict(
        binary=True, extrinsic=False,
        calibration_data=cal_data,
        cve_id=sample.cve_id, description=cve_descriptions.get(sample.cve_id, ''), domain=sample.domain_pddl,
    )

    prompt = eval_template.render(**render_args)
    print(f'--- binary intrinsic | {len(cal_data)} calibration examples | {len(prompt)} chars ---')
    print(prompt[:3000])
    if len(prompt) > 3000:
        print(f'\n... [{{len(prompt) - 3000}} chars truncated]')

interact(preview_intrinsic_binary, n_calibration=IntSlider(min=0, max=5, value=0, description='# Cal'));


interactive(children=(IntSlider(value=0, description='# Cal', max=5), Output()), _dom_classes=('widget-interac…

Because waiting more than 200ms didn’t work, add the free Qwen model to quickly validate the pipeline.

NVIDIA - "meta/llama-3.3-70b-instruct"

In [88]:
nvidia = OpenAI(
    api_key=os.getenv("NVIDIA_API_KEY"),
    base_url="https://integrate.api.nvidia.com/v1",
    timeout=120.0,
)
NVIDIA_MODEL = "meta/llama-3.3-70b-instruct"

t_start_llm_bin = time.time()
total_tokens_bin = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
PRICE_PER_1K_IN, PRICE_PER_1K_OUT = 0.0004, 0.0004  # llama-3.3-70b-instruct $/1K tokens
results_path = os.path.join(save_dir, "results_intrinsic_binary.jsonl")

open(results_path, "w").close()

# ── LLM intrinsic binary on reference data (incremental save) ──


# ── Test on test_samples (1 reference + 1 bad) ──
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    response, usage = llm_eval_intrinsic_binary(cve_id, description, domain_pddl, nvidia, NVIDIA_MODEL, seed=SEED)
    
    try:
        text = response.strip()
        if text.startswith("```"): text = text.split("\n", 1)[1].rsplit("```", 1)[0]
        raw_label = json.loads(text).get("label", "?")
        label = "True" if str(raw_label).lower() in ("true", "1", "yes") else "False"
    except Exception:
        label = "?"
    
    result = {"timestamp": datetime.now().isoformat(), "llm_model": NVIDIA_MODEL, "source": source, "cve_id": cve_id, "ap_id": ap_id, "label": label, "response_length": len(response), "parse_success": label != "?", "cost_usd": round((usage.get("prompt_tokens", 0) * PRICE_PER_1K_IN + usage.get("completion_tokens", 0) * PRICE_PER_1K_OUT) / 1000, 6), "llm_response": response, "usage": usage}
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    print(f"[{source}] {cve_id}/{ap_id:30s} | {label} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")



# llm_intrinsic_binary_results = []
# for entry in dataset:
#     for ap in entry["attack_paths"]:
#         response, usage = llm_eval_intrinsic_binary(entry["cve_id"], entry["description"], ap["domain"], nvidia, NVIDIA_MODEL, seed=SEED)
#         for k in total_tokens_bin: total_tokens_bin[k] += usage.get(k, 0)
#         
#         try:
#             text = response.strip()
#             if text.startswith("```"): text = text.split("\n", 1)[1].rsplit("```", 1)[0]
#             raw_label = json.loads(text).get("label", "?")
#             label = "True" if str(raw_label).lower() in ("true", "1", "yes") else "False"
#         except Exception:
#             label = "?"
#         
#         result = {"llm_model": NVIDIA_MODEL, "cve_id": entry["cve_id"], "ap_id": ap["ap_id"], "label": label, "llm_response": response, "usage": usage}
#         llm_intrinsic_binary_results.append(result)
#         
#         with open(results_path, "a") as f:
#             f.write(json.dumps(result) + "\n")
#         
#         print(f"{entry['cve_id']}/{ap['ap_id']:30s} | {label} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")
# 
# t_elapsed_llm_bin = time.time() - t_start_llm_bin
# 
# 
# print(f"\nDone: {len(llm_intrinsic_binary_results)} domains")
# 


[reference] CVE-2022-40149/AP2                            | True | 2.27s  in=4528 out=10
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 | True | 0.85s  in=5263 out=10


gpt-4.1-mini

In [89]:
#gpt-4.1-mini
load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    timeout=120.0,
)
GPT_MODEL = "gpt-4.1-mini"

t_start_llm_bin_gpt = time.time()
total_tokens_bin_gpt = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
PRICE_PER_1K_IN, PRICE_PER_1K_OUT = 0.0004, 0.0016  # gpt-4.1-mini $/1K tokens
results_path = os.path.join(save_dir, "results_intrinsic_binary.jsonl")

# NOTE: append to existing file (nvidia results already in it)

# ── LLM intrinsic binary on reference data (GPT, incremental save) ──


# ── Test on test_samples (1 reference + 1 bad) ──
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    response, usage = llm_eval_intrinsic_binary(cve_id, description, domain_pddl, openai_client, GPT_MODEL, seed=SEED)
    
    try:
        text = response.strip()
        if text.startswith("```"): text = text.split("\n", 1)[1].rsplit("```", 1)[0]
        raw_label = json.loads(text).get("label", "?")
        label = "True" if str(raw_label).lower() in ("true", "1", "yes") else "False"
    except Exception:
        label = "?"
    
    result = {"timestamp": datetime.now().isoformat(), "llm_model": GPT_MODEL, "source": source, "cve_id": cve_id, "ap_id": ap_id, "label": label, "response_length": len(response), "parse_success": label != "?", "cost_usd": round((usage.get("prompt_tokens", 0) * PRICE_PER_1K_IN + usage.get("completion_tokens", 0) * PRICE_PER_1K_OUT) / 1000, 6), "llm_response": response, "usage": usage}
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    print(f"[{source}] {cve_id}/{ap_id:30s} | {label} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")



# llm_intrinsic_binary_results_gpt = []
# for entry in dataset:
#     for ap in entry["attack_paths"]:
#         response, usage = llm_eval_intrinsic_binary(entry["cve_id"], entry["description"], ap["domain"], openai_client, GPT_MODEL, seed=SEED)
#         for k in total_tokens_bin_gpt: total_tokens_bin_gpt[k] += usage.get(k, 0)
#         
#         try:
#             text = response.strip()
#             if text.startswith("```"): text = text.split("\n", 1)[1].rsplit("```", 1)[0]
#             raw_label = json.loads(text).get("label", "?")
#             label = "True" if str(raw_label).lower() in ("true", "1", "yes") else "False"
#         except Exception:
#             label = "?"
#         
#         result = {"llm_model": GPT_MODEL, "cve_id": entry["cve_id"], "ap_id": ap["ap_id"], "label": label, "llm_response": response, "usage": usage}
#         llm_intrinsic_binary_results_gpt.append(result)
#         
#         with open(results_path, "a") as f:
#             f.write(json.dumps(result) + "\n")
#         
#         print(f"{entry['cve_id']}/{ap['ap_id']:30s} | {label} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")
# 
# t_elapsed_llm_bin_gpt = time.time() - t_start_llm_bin_gpt
# 
# print(f"\nDone: {len(llm_intrinsic_binary_results_gpt)} domains, {t_elapsed_llm_bin_gpt:.2f}s, Tokens: {total_tokens_bin_gpt}")
# 


[reference] CVE-2022-40149/AP2                            | True | 2.06s  in=4804 out=9
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 | True | 1.68s  in=5598 out=9


##### 3.3.1.2b Scored Intrinsic (12-criteria, integer 0-5)
`llm_eval_intrinsic_scored`: 12-criteria scored evaluation using `completion.md.jinja` (binary=False, extrinsic=False)


In [90]:
def llm_eval_intrinsic_scored(cve_id, description, domain, client, model, seed=42, n_calibration=2):
    """12-criteria scored evaluation (integer 0-5, violation/non-violation scale)."""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "intrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=False, extrinsic=False,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description, domain=domain,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=4096,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [91]:
def preview_intrinsic_scored(n_calibration=2):
    sample = few_shot_pool[0]
    cal_data = load_calibration_data(CALIBRATION_DATA_PATH, 'intrinsic', exclude_cve=sample.cve_id)[:n_calibration]
    render_args = dict(
        binary=False, extrinsic=False,
        calibration_data=cal_data,
        cve_id=sample.cve_id, description=cve_descriptions.get(sample.cve_id, ''), domain=sample.domain_pddl,
    )

    prompt = eval_template.render(**render_args)
    print(f'--- scored intrinsic | {len(cal_data)} calibration examples | {len(prompt)} chars ---')
    print(prompt[:3000])
    if len(prompt) > 3000:
        print(f'\n... [{{len(prompt) - 3000}} chars truncated]')

interact(preview_intrinsic_scored, n_calibration=IntSlider(min=0, max=5, value=0, description='# Cal'));


interactive(children=(IntSlider(value=0, description='# Cal', max=5), Output()), _dom_classes=('widget-interac…

Block 2: evaluation result parser and formatter

In [94]:
SCORED_CRITERIA = ["F1","F2","A1","A2","C1","C2","C3","V1","V2","V3","N1","N2"]

def parse_scored_response(response_text):
    """Parse LLM scored response JSON, extract scores."""
    try:
        text = response_text.strip()
        if text.startswith("```"):
            text = text.split("\n", 1)[1]
            text = text.rsplit("```", 1)[0]
        data = json.loads(text)
        return data
    except Exception:
        return None

def domain_min_score(scores, criteria=SCORED_CRITERIA):
    """Return min of valid criteria scores (0-5). None if no valid scores."""
    valid = [scores[k] for k in criteria if isinstance(scores.get(k), (int, float))]
    return min(valid) if valid else None

def summarize_intrinsic_scored(result):
    """Extract one-line summary from intrinsic scored result."""
    data = parse_scored_response(result["llm_response"])
    if data is None:
        return f"{result['cve_id']}/{result.get('ap_id', result.get('config', '?'))}  PARSE_ERROR"
    qmin = domain_min_score(data)
    scores_str = " | ".join(str(data.get(k, "?")) for k in SCORED_CRITERIA)
    ap = result.get("ap_id", result.get("config", "?"))
    return f"{result['cve_id']}/{ap:30s} | {scores_str} | min={qmin}"

def summarize_extrinsic_scored(result):
    """Extract one-line summary from extrinsic scored result (+ R1-R4)."""
    data = parse_scored_response(result["llm_response"])
    if data is None:
        return f"{result['cve_id']}/{result.get('generated', '?')} vs {result.get('reference', '?')}  PARSE_ERROR"
    ext_keys = SCORED_CRITERIA + ["R1","R2","R3","R4"]
    qmin = domain_min_score(data)
    scores_str = " | ".join(str(data.get(k, "?")) for k in ext_keys)
    gen = result.get("generated", result.get("ap_id", "?"))
    ref = result.get("reference", "?")
    return f"{result['cve_id']}/{gen:20s} vs {ref} | {scores_str} | min={qmin}"


Block 3: call the function with the code and the specifications from the data set

 NVIDIA_MODEL

In [95]:
t_start_llm_scored = time.time()
total_tokens_scored = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
PRICE_PER_1K_IN, PRICE_PER_1K_OUT = 0.0004, 0.0004  # llama-3.3-70b-instruct $/1K tokens
results_path = os.path.join(save_dir, "results_intrinsic_scored.jsonl")

open(results_path, "w").close()

# ── Test on test_samples (1 reference + 1 bad) ──
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    response, usage = llm_eval_intrinsic_scored(cve_id, description, domain_pddl, nvidia, NVIDIA_MODEL, seed=SEED)
    
    scores = parse_scored_response(response) or {"parse_error": True}
    qmin = domain_min_score(scores) if "parse_error" not in scores else None
    verdict = f"True({qmin})" if qmin is not None and qmin >= 3 else f"False({qmin})"
    
    result = {"timestamp": datetime.now().isoformat(), "llm_model": NVIDIA_MODEL, "source": source, "cve_id": cve_id, "ap_id": ap_id, "verdict": verdict, "min_score": qmin, "scores": scores, "response_length": len(response), "parse_success": "parse_error" not in scores, "cost_usd": round((usage.get("prompt_tokens", 0) * PRICE_PER_1K_IN + usage.get("completion_tokens", 0) * PRICE_PER_1K_OUT) / 1000, 6), "llm_response": response, "usage": usage}
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    print(f"[{source}] {cve_id}/{ap_id:30s} | {verdict} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")

# ── LLM intrinsic scored on reference data ──

# llm_intrinsic_scored_results = []
# for entry in dataset:
#     for ap in entry["attack_paths"]:
#         response, usage = llm_eval_intrinsic_scored(entry["cve_id"], entry["description"], ap["domain"], nvidia, NVIDIA_MODEL, seed=SEED)
#         for k in total_tokens_scored: total_tokens_scored[k] += usage.get(k, 0)
# 
#         scores = parse_scored_response(response) or {"parse_error": True}
#         qmin = domain_min_score(scores) if "parse_error" not in scores else None
#         verdict = f"True({qmin})" if qmin is not None and qmin >= 3 else f"False({qmin})"
# 
#         result = {"llm_model": NVIDIA_MODEL, "cve_id": entry["cve_id"], "ap_id": ap["ap_id"], "scores": scores, "min_score": qmin, "verdict": verdict, "llm_response": response, "usage": usage}
#         llm_intrinsic_scored_results.append(result)
# 
#         with open(results_path, "a") as f:
#             f.write(json.dumps(result) + "\n")
# 
#         print(f"{entry['cve_id']}/{ap['ap_id']:30s} | {verdict} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")
# 
# t_elapsed_llm_scored = time.time() - t_start_llm_scored
# 
# 
# print(f"\nDone: {len(llm_intrinsic_scored_results)} domains, {t_elapsed_llm_scored:.2f}s, Tokens: {total_tokens_scored}")
# 


[reference] CVE-2022-40149/AP2                            | True(5) | 2.69s  in=5585 out=103
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 | True(5) | 5.59s  in=6320 out=103


gpt-4.1-mini

In [96]:
load_dotenv(override=True)  # override=True Force overwrite existing environment variables.  
key = os.getenv("OPENAI_API_KEY", "")                                                                                   
print(f"Key loaded: {key[:8]}...{key[-4:]}" if len(key) > 12 else "Key NOT found or too short") 

Key loaded: sk-proj-...z4wA


In [97]:
# openai_client and GPT_MODEL already loaded in 3.3.1.2a binary intrinsic

t_start_llm_gpt = time.time()
total_tokens_gpt = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
PRICE_PER_1K_IN, PRICE_PER_1K_OUT = 0.0004, 0.0016  # gpt-4.1-mini $/1K tokens
results_path = os.path.join(save_dir, "results_intrinsic_scored.jsonl")

# NOTE: append to existing file (nvidia results already in it)

# ── LLM intrinsic scored on reference data (GPT, incremental save) ──


# ── Test on test_samples (1 reference + 1 bad) ──
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    response, usage = llm_eval_intrinsic_scored(cve_id, description, domain_pddl, openai_client, GPT_MODEL, seed=SEED)
    
    scores = parse_scored_response(response) or {"parse_error": True}
    qmin = domain_min_score(scores) if "parse_error" not in scores else None
    verdict = f"True({qmin})" if qmin is not None and qmin >= 3 else f"False({qmin})"
    
    result = {"timestamp": datetime.now().isoformat(), "llm_model": GPT_MODEL, "source": source, "cve_id": cve_id, "ap_id": ap_id, "verdict": verdict, "min_score": qmin, "scores": scores, "response_length": len(response), "parse_success": "parse_error" not in scores, "cost_usd": round((usage.get("prompt_tokens", 0) * PRICE_PER_1K_IN + usage.get("completion_tokens", 0) * PRICE_PER_1K_OUT) / 1000, 6), "llm_response": response, "usage": usage}
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    print(f"[{source}] {cve_id}/{ap_id:30s} | {verdict} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")



# llm_intrinsic_scored_results_gpt = []
# for entry in dataset:
#     for ap in entry["attack_paths"]:
#         response, usage = llm_eval_intrinsic_scored(entry["cve_id"], entry["description"], ap["domain"], openai_client, GPT_MODEL, seed=SEED)
#         for k in total_tokens_gpt: total_tokens_gpt[k] += usage.get(k, 0)
# 
#         scores = parse_scored_response(response) or {"parse_error": True}
#         qmin = domain_min_score(scores) if "parse_error" not in scores else None
#         verdict = f"True({qmin})" if qmin is not None and qmin >= 3 else f"False({qmin})"
# 
#         result = {"llm_model": GPT_MODEL, "cve_id": entry["cve_id"], "ap_id": ap["ap_id"], "scores": scores, "min_score": qmin, "verdict": verdict, "llm_response": response, "usage": usage}
#         llm_intrinsic_scored_results_gpt.append(result)
# 
#         with open(results_path, "a") as f:
#             f.write(json.dumps(result) + "\n")
# 
#         print(f"{entry['cve_id']}/{ap['ap_id']:30s} | {verdict} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")
# 
# t_elapsed_llm_gpt = time.time() - t_start_llm_gpt
# 
# 
# print(f"\nDone: {len(llm_intrinsic_scored_results_gpt)} domains, {t_elapsed_llm_gpt:.2f}s, Tokens: {total_tokens_gpt}")
# 


[reference] CVE-2022-40149/AP2                            | True(5) | 2.33s  in=5866 out=102
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 | True(5) | 2.16s  in=6660 out=102


#### 3.3.2 Extrinsic

##### 3.3.2.1a Embedding: reference PDDL vs generated PDDL Similarity

Block 1: the function to compute the embedding similarity between two blocks of PDDL code (domain + problem)

In [72]:
def embedding_similarity_extrinsic(pddl_texts_generated, pddl_texts_reference, model):
    """Cosine similarity matrix between generated and reference PDDL codes.
    Args:
        pddl_texts_generated: list of generated PDDL domain strings
        pddl_texts_reference: list of reference PDDL domain strings
        model: SentenceTransformer model
    Returns:
        numpy array of shape (len(generated), len(reference))
    """
    E_generated = model.encode(
        pddl_texts_generated,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    E_reference = model.encode(
        pddl_texts_reference,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    S = E_generated @ E_reference.T
    return S.float().cpu().numpy()

Block 2: Extrinsic cross-validation, performance estimation, and corrected proportion
Same process as intrinsic Block 5 but for extrinsic (domain vs domain):
1. Build positive/negative pairs: 
  - 1：CVE PDDL AP vs the PDDL APs of the same CVE 
  - 0：CVE PDDL AP vs the PDDL APs of the different CVE  
2. CVE-level GroupKFold CV with bootstrap threshold
3. Out-of-fold predictions → classification report → TPR, FPR
4. Apply threshold to reference domains

In [73]:
# ── Step 1: Build extrinsic positive/negative pairs ──

def build_extrinsic_pairs(dataset, model):
    """
    Build (scores, labels, groups) for extrinsic embedding evaluation.
    Positive (1): same CVE, different APs (domain_i vs domain_j)
    Negative (0): different CVE (domain_i vs domain_j)
    Groups: assigned by CVE of first domain in pair
    """
    # Collect all (domain, cve_id) pairs
    all_entries = []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            all_entries.append((ap["domain"], entry["cve_id"], ap["ap_id"]))

    domains = [p[0] for p in all_entries]
    cve_ids = [p[1] for p in all_entries]

    # Batch encode
    E = model.encode(domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    sim_matrix = (E @ E.T).float().cpu().numpy()

    scores, labels, groups = [], [], []
    n = len(all_entries)
    for i in range(n):
        for j in range(i+1, n):  # upper triangle only, symmetric
            sim = float(sim_matrix[i, j])
            label = 1 if cve_ids[i] == cve_ids[j] else 0
            scores.append(sim)
            labels.append(label)
            groups.append(cve_ids[i])  # group by first domain's CVE

    return np.array(scores), np.array(labels), np.array(groups)


ext_scores_a, ext_labels_a, ext_groups_a = build_extrinsic_pairs(dataset, embedding_model)
print(f"Extrinsic Embedding Model: {EMBEDDING_MODEL_NAME}")
print(f"  Positive pairs: {ext_labels_a.sum()}, Negative pairs: {(ext_labels_a == 0).sum()}")
print(f"  Groups (CVEs): {len(np.unique(ext_groups_a))}")


Extrinsic Embedding Model: all-MiniLM-L6-v2
  Positive pairs: 74, Negative pairs: 1411
  Groups (CVEs): 21


In [74]:
# ── Step 2: Extrinsic CV calibration ──

ext_cv_threshold_a, ext_cv_fold_thr_a, ext_cv_pred_a, ext_cv_true_a = run_calibration(
    ext_scores_a, ext_labels_a, ext_groups_a)
print(f"\nMedian threshold: {ext_cv_threshold_a:.4f}")
print("Fold thresholds:", [f"{t:.4f}" for t in ext_cv_fold_thr_a])


  Fold 1: threshold=0.9621, val CVEs=['CVE-2022-1471', 'CVE-2024-12798', 'CVE-2024-38809', 'CVE-2025-24813']
  Fold 2: threshold=0.9744, val CVEs=['CVE-2023-44487', 'CVE-2023-46589', 'CVE-2024-22262', 'CVE-2024-38820', 'CVE-2024-47072']
  Fold 3: threshold=0.9383, val CVEs=['CVE-2022-40150', 'CVE-2023-2976', 'CVE-2023-6378', 'CVE-2024-38816']
  Fold 4: threshold=0.9621, val CVEs=['CVE-2024-22243', 'CVE-2024-22259', 'CVE-2024-34447', 'CVE-2024-38286']
  Fold 5: threshold=0.9621, val CVEs=['CVE-2022-40149', 'CVE-2023-33202', 'CVE-2023-34055', 'CVE-2025-22228']

Median threshold: 0.9621
Fold thresholds: ['0.9621', '0.9744', '0.9383', '0.9621', '0.9621']


In [ ]:
# ── Step 3: Extrinsic performance report + save CV metrics ──
print(f"Extrinsic Embedding Model ({EMBEDDING_MODEL_NAME})")
print(classification_report(ext_cv_true_a, ext_cv_pred_a, zero_division=0))

ext_row_a = report_row(ext_cv_true_a, ext_cv_pred_a,
                      metric="embedding", model=EMBEDDING_MODEL_NAME,
                      mode="extrinsic", threshold=ext_cv_threshold_a)
print(f"TPR = {ext_row_a['tpr']:.4f}  FPR = {ext_row_a['fpr']:.4f}")

# ── Save CV calibration metrics (class 1 only) ──
save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_extrinsic_similarity.jsonl")

ext_cv_record = {
    "model": EMBEDDING_MODEL_NAME,
    "threshold": ext_cv_threshold_a,
    "fold_thresholds": ext_cv_fold_thr_a,
    "accuracy": ext_row_a.get("accuracy", float("nan")),
    "precision": ext_row_a.get("1__precision", float("nan")),
    "recall": ext_row_a.get("1__recall", float("nan")),
    "f1": ext_row_a.get("1__f1-score", float("nan")),
    "tpr": ext_row_a["tpr"],
    "fpr": ext_row_a["fpr"],
    "n_positive_pairs": int(ext_labels_a.sum()),
    "n_negative_pairs": int((ext_labels_a == 0).sum()),
}

existing = []
if os.path.exists(cv_path):
    with open(cv_path) as f:
        existing = [json.loads(line) for line in f if line.strip()]
    existing = [r for r in existing if r.get("model") != EMBEDDING_MODEL_NAME]
existing.append(ext_cv_record)
with open(cv_path, "w") as f:
    for r in existing:
        f.write(json.dumps(r) + "\n")


calibration_embedding_rows.append({
    "model": EMBEDDING_MODEL_NAME, "mode": "extrinsic", "threshold": ext_cv_threshold_a,
    "precision": ext_cv_record["precision"], "recall": ext_cv_record["recall"],
    "f1": ext_cv_record["f1"], "tpr": ext_row_a["tpr"], "fpr": ext_row_a["fpr"],
    "n_positive": int(ext_labels_a.sum()), "n_negative": int((ext_labels_a == 0).sum()),
})


Extrinsic Embedding Model (all-MiniLM-L6-v2)
              precision    recall  f1-score   support

           0       1.00      0.94      0.97      1411
           1       0.45      0.92      0.61        74

    accuracy                           0.94      1485
   macro avg       0.72      0.93      0.79      1485
weighted avg       0.97      0.94      0.95      1485

TPR = 0.9189  FPR = 0.0581


In [76]:
# ── Step 4: Apply extrinsic threshold to test samples ──

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, "results_extrinsic_similarity.jsonl")

open(results_path, "w").close()

# For extrinsic, compare each test sample against reference APs of the same CVE

print(f"Applying threshold {ext_cv_threshold_a:.4f} ({EMBEDDING_MODEL_NAME}) to test samples:")
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}

for source, cve_id, ap_id, domain_pddl in test_samples:
    ref_aps = ref_by_cve.get(cve_id, [])
    if not ref_aps:
        print(f"  [{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
        continue
    t0 = time.time()
    E_test = embedding_model.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
    ref_texts = [ap["domain"] for ap in ref_aps]
    E_ref = embedding_model.encode(ref_texts, convert_to_tensor=True, normalize_embeddings=True)
    sims = (E_test @ E_ref.T).float().cpu().numpy()[0]
    elapsed = time.time() - t0
    for j, ref_ap in enumerate(ref_aps):
        sim = float(sims[j])
        pred = "True" if sim >= ext_cv_threshold_a else "False"
        result = {"timestamp": datetime.now().isoformat(), 
            "source": source,
            "model": EMBEDDING_MODEL_NAME,
            "threshold": ext_cv_threshold_a,
            "cve_id": cve_id,
            "test_ap": ap_id,
            "ref_ap": ref_ap["ap_id"],
            "similarity": round(sim, 6),
            "prediction": pred,
            "elapsed_seconds": round(elapsed, 4),
        }
        with open(results_path, "a") as f:
            f.write(json.dumps(result) + "\n")
        print(f"  [{source}] {cve_id}/{ap_id} vs {ref_ap['ap_id']}  sim={sim:.4f}  pred={pred}  {elapsed:.3f}s")



# # ── Apply to ALL reference domain pairs (uncomment for full run) ──
# ext_ref_results_a = []
# for entry in dataset:
#     ref_aps = entry["attack_paths"]
#     if len(ref_aps) < 2:
#         continue
#     texts = [ap["domain"] for ap in ref_aps]
#     E = embedding_model.encode(texts, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
#     S = (E @ E.T).float().cpu().numpy()
#     for i in range(len(ref_aps)):
#         for j in range(i+1, len(ref_aps)):
#             sim = float(S[i, j])
#             pred = "True" if sim >= ext_cv_threshold_a else "False"
#             label = f"{entry['cve_id']}/{ref_aps[i]['ap_id']} vs {ref_aps[j]['ap_id']}"
#             ext_ref_results_a.append({"label": label, "similarity": sim, "prediction": pred})
#             print(f"  {label:45s} prediction: {pred} (similarity: {sim:.4f})")
#
# ext_ref_ppv_a = sum(r["prediction"] for r in ext_ref_results_a) / len(ext_ref_results_a) if ext_ref_results_a else 0
# print(f"  Extrinsic Reference PPV: {ext_ref_ppv_a:.4f}")
#
# with open(results_path, "w") as f:
#     for r in ext_ref_results_a:
#         r["model"] = EMBEDDING_MODEL_NAME
#         r["threshold"] = ext_cv_threshold_a
#         f.write(json.dumps(r) + "\n")


Applying threshold 0.9621 (all-MiniLM-L6-v2) to test samples:
  [reference] CVE-2022-40149/AP2 vs AP1  sim=1.0000  pred=True  0.409s
  [reference] CVE-2022-40149/AP2 vs AP2  sim=1.0000  pred=True  0.409s
  [bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 vs AP1  sim=1.0000  pred=True  0.246s


In [120]:
# ── 3.3.2.1a with second embedding model ──
# embedding_model_2 already loaded in intrinsic section

# Step 1: Build pairs
ext_scores_b, ext_labels_b, ext_groups_b = build_extrinsic_pairs(dataset, embedding_model_2)
print(f"Extrinsic Embedding Model: {EMBEDDING_MODEL_NAME_2}")
print(f"  Positive pairs: {ext_labels_b.sum()}, Negative pairs: {(ext_labels_b == 0).sum()}")

# Step 2: CV calibration
ext_cv_threshold_b, ext_cv_fold_thr_b, ext_cv_pred_b, ext_cv_true_b = run_calibration(
    ext_scores_b, ext_labels_b, ext_groups_b)
print(f"Median threshold: {ext_cv_threshold_b:.4f}")

# Step 3: Performance report + save CV metrics
print(f"\nExtrinsic Embedding Model ({EMBEDDING_MODEL_NAME_2})")
print(classification_report(ext_cv_true_b, ext_cv_pred_b, zero_division=0))

ext_row_b = report_row(ext_cv_true_b, ext_cv_pred_b,
                      metric="embedding", model=EMBEDDING_MODEL_NAME_2,
                      mode="extrinsic", threshold=ext_cv_threshold_b)
print(f"TPR = {ext_row_b['tpr']:.4f}  FPR = {ext_row_b['fpr']:.4f}")

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_extrinsic_similarity.jsonl")

ext_cv_record_b = {
    "model": EMBEDDING_MODEL_NAME_2,
    "threshold": ext_cv_threshold_b,
    "fold_thresholds": ext_cv_fold_thr_b,
    "accuracy": ext_row_b.get("accuracy", float("nan")),
    "precision": ext_row_b.get("1__precision", float("nan")),
    "recall": ext_row_b.get("1__recall", float("nan")),
    "f1": ext_row_b.get("1__f1-score", float("nan")),
    "tpr": ext_row_b["tpr"],
    "fpr": ext_row_b["fpr"],
    "n_positive_pairs": int(ext_labels_b.sum()),
    "n_negative_pairs": int((ext_labels_b == 0).sum()),
}

existing = []
if os.path.exists(cv_path):
    with open(cv_path) as f:
        existing = [json.loads(line) for line in f if line.strip()]
    existing = [r for r in existing if r.get("model") != EMBEDDING_MODEL_NAME_2]
existing.append(ext_cv_record_b)
with open(cv_path, "w") as f:
    for r in existing:
        f.write(json.dumps(r) + "\n")

calibration_embedding_rows.append({
    "model": EMBEDDING_MODEL_NAME_2, "mode": "extrinsic", "threshold": ext_cv_threshold_b,
    "precision": ext_cv_record_b["precision"], "recall": ext_cv_record_b["recall"],
    "f1": ext_cv_record_b["f1"], "tpr": ext_row_b["tpr"], "fpr": ext_row_b["fpr"],
    "n_positive": int(ext_labels_b.sum()), "n_negative": int((ext_labels_b == 0).sum()),
})

# Step 4: Apply to test samples
results_path = os.path.join(save_dir, "results_extrinsic_similarity.jsonl")

# Append
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}
print(f"\nApplying threshold {ext_cv_threshold_b:.4f} ({EMBEDDING_MODEL_NAME_2}) to test samples:")
for source, cve_id, ap_id, domain_pddl in test_samples:
    ref_aps = ref_by_cve.get(cve_id, [])
    if not ref_aps:
        print(f"  [{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
        continue
    t0 = time.time()
    E_test = embedding_model_2.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
    ref_texts = [ap["domain"] for ap in ref_aps]
    E_ref = embedding_model_2.encode(ref_texts, convert_to_tensor=True, normalize_embeddings=True)
    sims = (E_test @ E_ref.T).float().cpu().numpy()[0]
    elapsed = time.time() - t0
    for j, ref_ap in enumerate(ref_aps):
        sim = float(sims[j])
        pred_str = "True" if sim >= ext_cv_threshold_b else "False"
        result = {"timestamp": datetime.now().isoformat(), "source": source, "model": EMBEDDING_MODEL_NAME_2,
                  "threshold": ext_cv_threshold_b, "cve_id": cve_id, "test_ap": ap_id,
                  "ref_ap": ref_ap["ap_id"], "similarity": round(sim, 6),
                  "prediction": pred_str, "elapsed_seconds": round(elapsed, 4)}
        with open(results_path, "a") as f:
            f.write(json.dumps(result) + "\n")
        print(f"  [{source}] {cve_id}/{ap_id} vs {ref_ap['ap_id']}  sim={sim:.4f}  pred={pred_str}  {elapsed:.3f}s")


Extrinsic Embedding Model: BAAI/bge-base-en-v1.5
  Positive pairs: 74, Negative pairs: 1411
  Fold 1: threshold=0.9923, val CVEs=['CVE-2022-1471', 'CVE-2024-12798', 'CVE-2024-38809', 'CVE-2025-24813']
  Fold 2: threshold=0.9936, val CVEs=['CVE-2023-44487', 'CVE-2023-46589', 'CVE-2024-22262', 'CVE-2024-38820', 'CVE-2024-47072']
  Fold 3: threshold=0.9927, val CVEs=['CVE-2022-40150', 'CVE-2023-2976', 'CVE-2023-6378', 'CVE-2024-38816']
  Fold 4: threshold=0.9728, val CVEs=['CVE-2024-22243', 'CVE-2024-22259', 'CVE-2024-34447', 'CVE-2024-38286']
  Fold 5: threshold=0.9927, val CVEs=['CVE-2022-40149', 'CVE-2023-33202', 'CVE-2023-34055', 'CVE-2025-22228']
Median threshold: 0.9927

Extrinsic Embedding Model (BAAI/bge-base-en-v1.5)
              precision    recall  f1-score   support

           0       0.99      0.94      0.97      1411
           1       0.46      0.89      0.60        74

    accuracy                           0.94      1485
   macro avg       0.72      0.92      0.79      

##### 3.3.2.2 LLM as an Expert — Reference PDDL vs Candidate PDDL Match

##### 3.3.2.2a Binary Extrinsic (True/False)
`llm_eval_extrinsic_binary`: binary classification using `completion.md.jinja` (binary=True, extrinsic=True)


In [98]:
def llm_eval_extrinsic_binary(cve_id, description, domain_reference, domain_generated, client, model, seed=42, n_calibration=2):
    """Binary True/False: does the candidate match the CVE and reference?"""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "extrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=True, extrinsic=True,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description,
        domain=domain_generated, domain_reference=domain_reference,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=512,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


Block 2: call the function with the code and the specifications from the data set

In [99]:
def preview_extrinsic_binary(n_calibration=2):
    sample = few_shot_pool[0]
    cal_data = load_calibration_data(CALIBRATION_DATA_PATH, 'extrinsic', exclude_cve=sample.cve_id)[:n_calibration]
    render_args = dict(
        binary=True, extrinsic=True,
        calibration_data=cal_data,
        cve_id=sample.cve_id, description=cve_descriptions.get(sample.cve_id, ''), domain=sample.domain_pddl,
    )
    ref = few_shot_pool[1] if len(few_shot_pool) > 1 else few_shot_pool[0]
    render_args['domain_reference'] = ref.domain_pddl
    prompt = eval_template.render(**render_args)
    print(f'--- binary extrinsic | {len(cal_data)} calibration examples | {len(prompt)} chars ---')
    print(prompt[:3000])
    if len(prompt) > 3000:
        print(f'\n... [{{len(prompt) - 3000}} chars truncated]')

interact(preview_extrinsic_binary, n_calibration=IntSlider(min=0, max=5, value=0, description='# Cal'));


interactive(children=(IntSlider(value=0, description='# Cal', max=5), Output()), _dom_classes=('widget-interac…

In [ ]:
t_start_ext_bin = time.time()
total_tokens_ext_bin = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
PRICE_PER_1K_IN, PRICE_PER_1K_OUT = 0.0004, 0.0004  # llama-3.3-70b-instruct $/1K tokens
results_path = os.path.join(save_dir, "results_extrinsic_binary.jsonl")

open(results_path, "w").close()

# ── Test on test_samples (1 reference + 1 bad) ──
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    ref_aps = ref_by_cve.get(cve_id, [])
    if not ref_aps:
        print(f"[{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
        continue
    for ref_ap in ref_aps:
        response, usage = llm_eval_extrinsic_binary(cve_id, description, ref_ap["domain"], domain_pddl, nvidia, NVIDIA_MODEL, seed=SEED)
        
        try:
            text = response.strip()
            if text.startswith("```"): text = text.split("\n", 1)[1].rsplit("```", 1)[0]
            raw_label = json.loads(text).get("label", "?")
            label = "True" if str(raw_label).lower() in ("true", "1", "yes") else "False"
        except Exception:
            label = "?"
        
        result = {"timestamp": datetime.now().isoformat(), "llm_model": NVIDIA_MODEL, "source": source, "cve_id": cve_id, "generated": ap_id, "reference": ref_ap["ap_id"], "label": label, "response_length": len(response), "parse_success": label != "?", "cost_usd": round((usage.get("prompt_tokens", 0) * PRICE_PER_1K_IN + usage.get("completion_tokens", 0) * PRICE_PER_1K_OUT) / 1000, 6), "llm_response": response, "usage": usage}
        with open(results_path, "a") as f:
            f.write(json.dumps(result) + "\n")
        print(f"[{source}] {cve_id}/{ap_id:20s} vs {ref_ap['ap_id']} | {label} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")


# ── LLM extrinsic binary on reference data ──

# llm_extrinsic_binary_results = []
# for entry in dataset:
#     ref_aps = entry["attack_paths"]
#     if len(ref_aps) < 2:
#         continue
#     for i, ap_gen in enumerate(ref_aps):
#         for j, ap_ref in enumerate(ref_aps):
#             response, usage = llm_eval_extrinsic_binary(
#                 entry["cve_id"], entry["description"], ap_ref["domain"],
#                 ap_gen["domain"], nvidia, NVIDIA_MODEL, seed=SEED)
#             for k in total_tokens_ext_bin: total_tokens_ext_bin[k] += usage.get(k, 0)
#             
#             try:
#                 text = response.strip()
#                 if text.startswith("```"): text = text.split("\n", 1)[1].rsplit("```", 1)[0]
#                 raw_label = json.loads(text).get("label", "?")
#                 label = "True" if str(raw_label).lower() in ("true", "1", "yes") else "False"
#             except Exception:
#                 label = "?"
#             
#             result = {
#                 "llm_model": NVIDIA_MODEL, "cve_id": entry["cve_id"], "generated": ap_gen["ap_id"],
#                 "reference": ap_ref["ap_id"], "label": label, "llm_response": response, "usage": usage,
#             }
#             llm_extrinsic_binary_results.append(result)
#             
#             with open(results_path, "a") as f:
#                 f.write(json.dumps(result) + "\n")
#             
#             print(f"{entry['cve_id']}/{ap_gen['ap_id']:20s} vs {ap_ref['ap_id']} | {label} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")
# 
# t_elapsed_ext_bin = time.time() - t_start_ext_bin
# 
# 
# print(f"\nDone: {len(llm_extrinsic_binary_results)} pairs, {t_elapsed_ext_bin:.2f}s, Tokens: {total_tokens_ext_bin}")
# 


[reference] CVE-2022-40149/AP2                  vs AP1 | False | 3.13s  in=7493 out=10
[reference] CVE-2022-40149/AP2                  vs AP2 | True | 1.13s  in=7738 out=10
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 vs AP1 | False | 1.27s  in=7567 out=10


gpt-4.1-mini extrinsic binary

In [101]:
#gpt-4.1-mini extrinsic binary

t_start_ext_bin_gpt = time.time()
total_tokens_ext_bin_gpt = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
PRICE_PER_1K_IN, PRICE_PER_1K_OUT = 0.0004, 0.0016  # gpt-4.1-mini $/1K tokens
results_path = os.path.join(save_dir, "results_extrinsic_binary.jsonl")


# ── Test on test_samples (1 reference + 1 bad) ──
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    ref_aps = ref_by_cve.get(cve_id, [])
    if not ref_aps:
        print(f"[{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
        continue
    for ref_ap in ref_aps:
        response, usage = llm_eval_extrinsic_binary(cve_id, description, ref_ap["domain"], domain_pddl, openai_client, GPT_MODEL, seed=SEED)
        
        try:
            text = response.strip()
            if text.startswith("```"): text = text.split("\n", 1)[1].rsplit("```", 1)[0]
            raw_label = json.loads(text).get("label", "?")
            label = "True" if str(raw_label).lower() in ("true", "1", "yes") else "False"
        except Exception:
            label = "?"
        
        result = {"timestamp": datetime.now().isoformat(), "llm_model": GPT_MODEL, "source": source, "cve_id": cve_id, "generated": ap_id, "reference": ref_ap["ap_id"], "label": label, "response_length": len(response), "parse_success": label != "?", "cost_usd": round((usage.get("prompt_tokens", 0) * PRICE_PER_1K_IN + usage.get("completion_tokens", 0) * PRICE_PER_1K_OUT) / 1000, 6), "llm_response": response, "usage": usage}
        with open(results_path, "a") as f:
            f.write(json.dumps(result) + "\n")
        print(f"[{source}] {cve_id}/{ap_id:20s} vs {ref_ap['ap_id']} | {label} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")


# ── LLM extrinsic binary on reference data ──
# llm_extrinsic_binary_results_gpt = []
# for entry in dataset:
#     ref_aps = entry["attack_paths"]
#     if len(ref_aps) < 2:
#         continue
#     for i, ap_gen in enumerate(ref_aps):
#         for j, ap_ref in enumerate(ref_aps):
#             response, usage = llm_eval_extrinsic_binary(
#                 entry["cve_id"], entry["description"], ap_ref["domain"],
#                 ap_gen["domain"], openai_client, GPT_MODEL, seed=SEED)
#             for k in total_tokens_ext_bin_gpt: total_tokens_ext_bin_gpt[k] += usage.get(k, 0)
#             
#             try:
#                 text = response.strip()
#                 if text.startswith("```"): text = text.split("\n", 1)[1].rsplit("```", 1)[0]
#                 raw_label = json.loads(text).get("label", "?")
#                 label = "True" if str(raw_label).lower() in ("true", "1", "yes") else "False"
#             except Exception:
#                 label = "?"
#             
#             result = {
#                 "llm_model": GPT_MODEL, "cve_id": entry["cve_id"], "generated": ap_gen["ap_id"],
#                 "reference": ap_ref["ap_id"], "label": label, "llm_response": response, "usage": usage,
#             }
#             llm_extrinsic_binary_results_gpt.append(result)
#             
#             with open(results_path, "a") as f:
#                 f.write(json.dumps(result) + "\n")
#             
#             print(f"{entry['cve_id']}/{ap_gen['ap_id']:20s} vs {ap_ref['ap_id']} | {label} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")
# 
# t_elapsed_ext_bin_gpt = time.time() - t_start_ext_bin_gpt
# 
# print(f"\nDone: {len(llm_extrinsic_binary_results_gpt)} pairs, {t_elapsed_ext_bin_gpt:.2f}s, Tokens: {total_tokens_ext_bin_gpt}")
# 


[reference] CVE-2022-40149/AP2                  vs AP1 | True | 1.12s  in=8027 out=9
[reference] CVE-2022-40149/AP2                  vs AP2 | True | 0.81s  in=8287 out=9
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 vs AP1 | True | 1.25s  in=8142 out=9


##### 3.3.2.2b Scored Extrinsic (16-criteria, integer 0-5)
`llm_eval_extrinsic_scored`: 16-criteria scored evaluation (12 quality + 4 alignment) using `completion.md.jinja` (binary=False, extrinsic=True)


In [102]:
def llm_eval_extrinsic_scored(cve_id, description, domain_reference, domain_generated, client, model, seed=42, n_calibration=2):
    """16-criteria scored evaluation (12 quality + 4 reference-comparison, integer 0-5)."""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "extrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=False, extrinsic=True,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description,
        domain=domain_generated, domain_reference=domain_reference,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=4096,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [103]:
def preview_extrinsic_scored(n_calibration=2):
    sample = few_shot_pool[0]
    cal_data = load_calibration_data(CALIBRATION_DATA_PATH, 'extrinsic', exclude_cve=sample.cve_id)[:n_calibration]
    render_args = dict(
        binary=False, extrinsic=True,
        calibration_data=cal_data,
        cve_id=sample.cve_id, description=cve_descriptions.get(sample.cve_id, ''), domain=sample.domain_pddl,
    )
    ref = few_shot_pool[1] if len(few_shot_pool) > 1 else few_shot_pool[0]
    render_args['domain_reference'] = ref.domain_pddl
    prompt = eval_template.render(**render_args)
    print(f'--- scored extrinsic | {len(cal_data)} calibration examples | {len(prompt)} chars ---')
    print(prompt[:3000])
    if len(prompt) > 3000:
        print(f'\n... [{{len(prompt) - 3000}} chars truncated]')

interact(preview_extrinsic_scored, n_calibration=IntSlider(min=0, max=5, value=0, description='# Cal'));


interactive(children=(IntSlider(value=0, description='# Cal', max=5), Output()), _dom_classes=('widget-interac…

Block 2: call the function with the code and the specifications from the data set

NVIDIA_MODEL

In [106]:
t_start_ext_scored = time.time()
total_tokens_ext_scored = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
PRICE_PER_1K_IN, PRICE_PER_1K_OUT = 0.0004, 0.0004  # llama-3.3-70b-instruct $/1K tokens
results_path = os.path.join(save_dir, "results_extrinsic_scored.jsonl")

EXT_SCORED_CRITERIA = SCORED_CRITERIA + ["R1","R2","R3","R4"]

open(results_path, "w").close()


# ── Test on test_samples (1 reference + 1 bad) ──
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    ref_aps = ref_by_cve.get(cve_id, [])
    if not ref_aps:
        print(f"[{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
        continue
    for ref_ap in ref_aps:
        response, usage = llm_eval_extrinsic_scored(cve_id, description, ref_ap["domain"], domain_pddl, nvidia, NVIDIA_MODEL, seed=SEED)
        
        scores = parse_scored_response(response) or {"parse_error": True}
        if "parse_error" not in scores:
            quality_min = domain_min_score(scores)  # F1-N2
            alignment_min = domain_min_score(scores, criteria=["R1","R2","R3","R4"])  # R1-R4
            qmin = min(quality_min, alignment_min) if quality_min is not None and alignment_min is not None else None
        else:
            quality_min, alignment_min, qmin = None, None, None
        verdict = f"True({qmin})" if qmin is not None and qmin >= 3 else f"False({qmin})"
        
        result = {"timestamp": datetime.now().isoformat(), "llm_model": NVIDIA_MODEL, "source": source, "cve_id": cve_id, "generated": ap_id, "reference": ref_ap["ap_id"], "verdict": verdict, "min_score": qmin, "quality_min": quality_min, "alignment_min": alignment_min, "scores": scores, "response_length": len(response), "parse_success": "parse_error" not in scores, "cost_usd": round((usage.get("prompt_tokens", 0) * PRICE_PER_1K_IN + usage.get("completion_tokens", 0) * PRICE_PER_1K_OUT) / 1000, 6), "llm_response": response, "usage": usage}
        with open(results_path, "a") as f:
            f.write(json.dumps(result) + "\n")
        print(f"[{source}] {cve_id}/{ap_id:20s} vs {ref_ap['ap_id']} | {verdict} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")


# ── LLM extrinsic scored on reference data ──
# llm_extrinsic_scored_results = []
# for entry in dataset:
#     ref_aps = entry["attack_paths"]
#     if len(ref_aps) < 2:
#         continue
#     for i, ap_gen in enumerate(ref_aps):
#         for j, ap_ref in enumerate(ref_aps):
#             response, usage = llm_eval_extrinsic_scored(
#                 entry["cve_id"], entry["description"], ap_ref["domain"],
#                 ap_gen["domain"], nvidia, NVIDIA_MODEL, seed=SEED)
#             for k in total_tokens_ext_scored: total_tokens_ext_scored[k] += usage.get(k, 0)
# 
#             scores = parse_scored_response(response) or {"parse_error": True}
#             qmin = domain_min_score(scores) if "parse_error" not in scores else None
#             verdict = f"True({qmin})" if qmin is not None and qmin >= 3 else f"False({qmin})"
# 
#             result = {
#                 "llm_model": NVIDIA_MODEL, "cve_id": entry["cve_id"], "generated": ap_gen["ap_id"],
#                 "reference": ap_ref["ap_id"], "scores": scores, "min_score": qmin, "verdict": verdict,
#                 "llm_response": response, "usage": usage,
#             }
#             llm_extrinsic_scored_results.append(result)
# 
#             with open(results_path, "a") as f:
#                 f.write(json.dumps(result) + "\n")
# 
#             print(f"{entry['cve_id']}/{ap_gen['ap_id']:20s} vs {ap_ref['ap_id']} | {verdict} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")
# 
# t_elapsed_ext_scored = time.time() - t_start_ext_scored
# 
# 
# print(f"\nDone: {len(llm_extrinsic_scored_results)} pairs, {t_elapsed_ext_scored:.2f}s, Tokens: {total_tokens_ext_scored}")
# 


[reference] CVE-2022-40149/AP2                  vs AP1 | True(4) | 7.22s  in=8856 out=135
[reference] CVE-2022-40149/AP2                  vs AP2 | True(4) | 10.5s  in=9101 out=135
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 vs AP1 | True(5) | 5.93s  in=8930 out=135


gpt-4.1-mini

In [107]:
t_start_ext_gpt = time.time()
total_tokens_ext_gpt = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "llm-as-experts")
os.makedirs(save_dir, exist_ok=True)
PRICE_PER_1K_IN, PRICE_PER_1K_OUT = 0.0004, 0.0016  # gpt-4.1-mini $/1K tokens
results_path = os.path.join(save_dir, "results_extrinsic_scored.jsonl")


# ── Test on test_samples (1 reference + 1 bad) ──
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    ref_aps = ref_by_cve.get(cve_id, [])
    if not ref_aps:
        print(f"[{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
        continue
    for ref_ap in ref_aps:
        response, usage = llm_eval_extrinsic_scored(cve_id, description, ref_ap["domain"], domain_pddl, openai_client, GPT_MODEL, seed=SEED)
        
        scores = parse_scored_response(response) or {"parse_error": True}
        if "parse_error" not in scores:
            quality_min = domain_min_score(scores)  # F1-N2
            alignment_min = domain_min_score(scores, criteria=["R1","R2","R3","R4"])  # R1-R4
            qmin = min(quality_min, alignment_min) if quality_min is not None and alignment_min is not None else None
        else:
            quality_min, alignment_min, qmin = None, None, None
        verdict = f"True({qmin})" if qmin is not None and qmin >= 3 else f"False({qmin})"
        
        result = {"timestamp": datetime.now().isoformat(), "llm_model": GPT_MODEL, "source": source, "cve_id": cve_id, "generated": ap_id, "reference": ref_ap["ap_id"], "verdict": verdict, "min_score": qmin, "quality_min": quality_min, "alignment_min": alignment_min, "scores": scores, "response_length": len(response), "parse_success": "parse_error" not in scores, "cost_usd": round((usage.get("prompt_tokens", 0) * PRICE_PER_1K_IN + usage.get("completion_tokens", 0) * PRICE_PER_1K_OUT) / 1000, 6), "llm_response": response, "usage": usage}
        with open(results_path, "a") as f:
            f.write(json.dumps(result) + "\n")
        print(f"[{source}] {cve_id}/{ap_id:20s} vs {ref_ap['ap_id']} | {verdict} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")


# ── LLM extrinsic scored on reference data ──
# llm_extrinsic_scored_results_gpt = []
# for entry in dataset:
#     ref_aps = entry["attack_paths"]
#     if len(ref_aps) < 2:
#         continue
#     for i, ap_gen in enumerate(ref_aps):
#         for j, ap_ref in enumerate(ref_aps):
#             if i == j:
#                 continue
#             response, usage = llm_eval_extrinsic_scored(
#                 entry["cve_id"], entry["description"], ap_ref["domain"],
#                 ap_gen["domain"], openai_client, GPT_MODEL, seed=SEED)
#             for k in total_tokens_ext_gpt: total_tokens_ext_gpt[k] += usage.get(k, 0)
# 
#             scores = parse_scored_response(response) or {"parse_error": True}
#             qmin = domain_min_score(scores) if "parse_error" not in scores else None
#             verdict = f"True({qmin})" if qmin is not None and qmin >= 3 else f"False({qmin})"
# 
#             result = {
#                 "llm_model": GPT_MODEL, "cve_id": entry["cve_id"], "generated": ap_gen["ap_id"],
#                 "reference": ap_ref["ap_id"], "scores": scores, "min_score": qmin, "verdict": verdict,
#                 "llm_response": response, "usage": usage,
#             }
#             llm_extrinsic_scored_results_gpt.append(result)
# 
#             with open(results_path, "a") as f:
#                 f.write(json.dumps(result) + "\n")
# 
#             print(f"{entry['cve_id']}/{ap_gen['ap_id']:20s} vs {ap_ref['ap_id']} | {verdict} | {usage.get('elapsed_seconds', '')}s  in={usage.get('prompt_tokens', '?')} out={usage.get('completion_tokens', '?')}")
# 
# t_elapsed_ext_gpt = time.time() - t_start_ext_gpt
# 
# 
# print(f"\nDone: {len(llm_extrinsic_scored_results_gpt)} pairs, {t_elapsed_ext_gpt:.2f}s, Tokens: {total_tokens_ext_gpt}")
# 


[reference] CVE-2022-40149/AP2                  vs AP1 | True(5) | 17.42s  in=9395 out=856
[reference] CVE-2022-40149/AP2                  vs AP2 | True(5) | 5.83s  in=9655 out=134
[bad] CVE-2024-38809/AP1_inject_capability_violation_rep1 vs AP1 | True(3) | 16.38s  in=9510 out=751


## 4. Inter-Rater Agreement (Fleiss' Kappa)

Compute Fleiss' kappa between LLM raters (nvidia llama-3.3-70b, gpt-4.1-mini) on binary labels.
Each domain is rated True/False by each model. Fleiss' kappa measures agreement beyond chance.


In [ ]:
def load_jsonl(fpath):
    if not os.path.exists(fpath):
        return []
    with open(fpath) as f:
        return [json.loads(line) for line in f if line.strip()]

# ── Load all results ──
results_dir_i = os.path.join(RESULTS_BASE, "semantic", "intrinsic")
results_dir_e = os.path.join(RESULTS_BASE, "semantic", "extrinsic")

syntax_data = load_jsonl(os.path.join(RESULTS_BASE, "syntax", "results_syntax.jsonl"))
solv_data = load_jsonl(os.path.join(RESULTS_BASE, "solvability", "results_solvability.jsonl"))
intr_sim_data = load_jsonl(os.path.join(results_dir_i, "similarity", "results_intrinsic_similarity.jsonl"))
intr_bin_data = load_jsonl(os.path.join(results_dir_i, "llm-as-experts", "results_intrinsic_binary.jsonl"))
intr_scored_data = load_jsonl(os.path.join(results_dir_i, "llm-as-experts", "results_intrinsic_scored.jsonl"))
extr_sim_data = load_jsonl(os.path.join(results_dir_e, "similarity", "results_extrinsic_similarity.jsonl"))
extr_bin_data = load_jsonl(os.path.join(results_dir_e, "llm-as-experts", "results_extrinsic_binary.jsonl"))
extr_scored_data = load_jsonl(os.path.join(results_dir_e, "llm-as-experts", "results_extrinsic_scored.jsonl"))

def to_binary(val):
    """Normalize any label/verdict to 1 (True) or 0 (False). Returns None if unparseable."""
    s = str(val).strip()
    if s.startswith("True") or s in ("true", "1", "yes"):
        return 1
    if s.startswith("False") or s in ("false", "0", "no"):
        return 0
    return None

def domain_key(d):
    return (d.get("cve_id", ""), d.get("ap_id", ""))

# ── Build rating matrix: rows=domains, columns=raters ──
# Raters: syntax, solvability, intrinsic_sim, intr_bin_{model}, intr_scored_{model},
#         extrinsic_sim, extr_bin_{model}, extr_scored_{model}

ratings = defaultdict(dict)  # {domain_key: {rater_name: 0/1}}

for d in syntax_data:
    k = domain_key(d)
    ratings[k]["syntax"] = to_binary(d.get("syntax_ok"))

for d in solv_data:
    k = domain_key(d)
    ratings[k]["solvability"] = to_binary(d.get("solvable"))

for d in intr_sim_data:
    k = domain_key(d)
    model = d.get("model", "unknown")
    short = model_short_name(model)
    ratings[k][f"intr_sim_{short}"] = to_binary(d.get("prediction"))

for d in intr_bin_data:
    k = domain_key(d)
    model = d.get("llm_model", "")
    short = model_short_name(model)
    ratings[k][f"intr_bin_{short}"] = to_binary(d.get("label"))

for d in intr_scored_data:
    k = domain_key(d)
    model = d.get("llm_model", "")
    short = model_short_name(model)
    ratings[k][f"intr_scored_{short}"] = to_binary(d.get("verdict"))

# Extrinsic: take worst across ref APs per domain
extr_sim_agg = defaultdict(lambda: defaultdict(list))
for d in extr_sim_data:
    k = (d.get("cve_id", ""), d.get("test_ap", d.get("ap_id", "")))
    model = d.get("model", "unknown")
    short = model_short_name(model)
    extr_sim_agg[k][short].append(to_binary(d.get("prediction")))
for k, model_vals in extr_sim_agg.items():
    for short, vals in model_vals.items():
        valid = [v for v in vals if v is not None]
        ratings[k][f"extr_sim_{short}"] = min(valid) if valid else None  # worst case

extr_bin_agg = defaultdict(lambda: defaultdict(list))
for d in extr_bin_data:
    k = (d.get("cve_id", ""), d.get("generated", ""))
    model = d.get("llm_model", "")
    short = model_short_name(model)
    extr_bin_agg[k][short].append(to_binary(d.get("label")))
for k, model_vals in extr_bin_agg.items():
    for short, vals in model_vals.items():
        valid = [v for v in vals if v is not None]
        ratings[k][f"extr_bin_{short}"] = min(valid) if valid else None

extr_scored_agg = defaultdict(lambda: defaultdict(list))
for d in extr_scored_data:
    k = (d.get("cve_id", ""), d.get("generated", ""))
    model = d.get("llm_model", "")
    short = model_short_name(model)
    extr_scored_agg[k][short].append(to_binary(d.get("verdict")))
for k, model_vals in extr_scored_agg.items():
    for short, vals in model_vals.items():
        valid = [v for v in vals if v is not None]
        ratings[k][f"extr_scored_{short}"] = min(valid) if valid else None

# ── Determine all raters present ──
all_raters = sorted(set(r for v in ratings.values() for r in v))
print(f"Raters ({len(all_raters)}): {all_raters}")

# ── Build matrix: only domains with ALL raters ──
matrix = []
domain_keys_used = []
for k, r in sorted(ratings.items()):
    row = [r.get(rater) for rater in all_raters]
    if all(v is not None for v in row):
        matrix.append(row)
        domain_keys_used.append(k)

print(f"Domains with all raters: {len(matrix)} / {len(ratings)}")

if len(matrix) >= 2 and len(all_raters) >= 2:
    matrix_np = np.array(matrix)
    agg_table, _ = aggregate_raters(matrix_np)
    kappa = _fleiss_kappa(agg_table, method="fleiss")

    if kappa < 0: interp = "Poor"
    elif kappa < 0.20: interp = "Slight"
    elif kappa < 0.40: interp = "Fair"
    elif kappa < 0.60: interp = "Moderate"
    elif kappa < 0.80: interp = "Substantial"
    else: interp = "Almost Perfect"

    print(f"\nFleiss' Kappa = {kappa:.4f}  ({interp})")
    print(f"  Subjects: {len(matrix)}, Raters: {len(all_raters)}")

    # Per-domain detail
    print(f"\nPer-domain ratings:")
    for i, k in enumerate(domain_keys_used):
        row = matrix[i]
        agree = "AGREE" if len(set(row)) == 1 else "DISAGREE"
        votes = dict(zip(all_raters, row))
        true_count = sum(row)
        print(f"  {k[0]}/{k[1]:30s} | {true_count}/{len(row)} True | {agree}")

    # Save
    kappa_result = {
        "timestamp": datetime.now().isoformat(),
        "kappa": round(kappa, 4),
        "interpretation": interp,
        "n_subjects": len(matrix),
        "n_raters": len(all_raters),
        "raters": all_raters,
    }
    kappa_path = os.path.join(RESULTS_BASE, "agreement", "fleiss_kappa.jsonl")
    os.makedirs(os.path.dirname(kappa_path), exist_ok=True)
    with open(kappa_path, "w") as f:
        f.write(json.dumps(kappa_result) + "\n")
else:
    print("Insufficient data for Fleiss' kappa (need >= 2 domains with all raters)")
    kappa_result = None


Raters (14): ['extr_bin_gpt-4.1-mini', 'extr_bin_llama-3.3-70b', 'extr_scored_gpt-4.1-mini', 'extr_scored_llama-3.3-70b', 'extr_sim_all-MiniLM-L6-v2', 'extr_sim_bge-base-en-v1.5', 'intr_bin_gpt-4.1-mini', 'intr_bin_llama-3.3-70b', 'intr_scored_gpt-4.1-mini', 'intr_scored_llama-3.3-70b', 'intr_sim_all-MiniLM-L6-v2', 'intr_sim_bge-base-en-v1.5', 'solvability', 'syntax']
Domains with all raters: 2 / 2

Fleiss' Kappa = -0.0676  (Poor)
  Subjects: 2, Raters: 14

Per-domain ratings:
  CVE-2022-40149/AP2                            | 11/14 True | DISAGREE
  CVE-2024-38809/AP1_inject_capability_violation_rep1 | 12/14 True | DISAGREE


## 5. Evaluation Summary
Aggregate all evaluation results into one row per domain.


In [139]:
import pandas as pd
from collections import defaultdict

# ── Load all result files ──
def load_jsonl(fpath):
    if not os.path.exists(fpath):
        return []
    with open(fpath) as f:
        return [json.loads(line) for line in f if line.strip()]

results_dir_i = os.path.join(RESULTS_BASE, "semantic", "intrinsic")
results_dir_e = os.path.join(RESULTS_BASE, "semantic", "extrinsic")

syntax_data = load_jsonl(os.path.join(RESULTS_BASE, "syntax", "results_syntax.jsonl"))
solv_data = load_jsonl(os.path.join(RESULTS_BASE, "solvability", "results_solvability.jsonl"))
intr_sim_data = load_jsonl(os.path.join(results_dir_i, "similarity", "results_intrinsic_similarity.jsonl"))
intr_bin_data = load_jsonl(os.path.join(results_dir_i, "llm-as-experts", "results_intrinsic_binary.jsonl"))
intr_scored_data = load_jsonl(os.path.join(results_dir_i, "llm-as-experts", "results_intrinsic_scored.jsonl"))
extr_sim_data = load_jsonl(os.path.join(results_dir_e, "similarity", "results_extrinsic_similarity.jsonl"))
extr_bin_data = load_jsonl(os.path.join(results_dir_e, "llm-as-experts", "results_extrinsic_binary.jsonl"))
extr_scored_data = load_jsonl(os.path.join(results_dir_e, "llm-as-experts", "results_extrinsic_scored.jsonl"))
kappa_data = load_jsonl(os.path.join(RESULTS_BASE, "agreement", "fleiss_kappa.jsonl"))

def domain_key(d):
    return (d.get("cve_id", ""), d.get("ap_id", ""))

# ── Build per-domain summary ──
summary = defaultdict(dict)
all_time_keys = []
all_token_keys = []
all_cost_keys = []

for d in syntax_data:
    k = domain_key(d)
    summary[k]["timestamp"] = d.get("timestamp", "")
    summary[k]["source"] = d.get("source", "")
    summary[k]["cve_id"] = d.get("cve_id", "")
    summary[k]["ap_id"] = d.get("ap_id", "")
    summary[k]["syntax"] = "True" if d.get("syntax_ok") else "False"
    summary[k]["syntax_time"] = d.get("elapsed_seconds", 0)

for d in solv_data:
    k = domain_key(d)
    summary[k]["solvability"] = "True" if d.get("solvable") else "False"
    summary[k]["solvability_time"] = d.get("elapsed_seconds", 0)

for d in intr_sim_data:
    k = domain_key(d)
    model = model_short_name(d.get("model", "unknown"))
    summary[k][f"intr_sim_{model}"] = d.get("prediction", "")
    summary[k][f"intr_sim_{model}_time"] = d.get("elapsed_seconds", 0)

for d in intr_bin_data:
    k = domain_key(d)
    short = model_short_name(d.get("llm_model", ""))
    summary[k][f"intr_bin_{short}"] = d.get("label", "")
    summary[k][f"intr_bin_{short}_time"] = d.get("usage", {}).get("elapsed_seconds", 0)
    summary[k][f"intr_bin_{short}_tokens"] = d.get("usage", {}).get("total_tokens", 0)
    summary[k][f"intr_bin_{short}_cost"] = d.get("cost_usd", 0)

for d in intr_scored_data:
    k = domain_key(d)
    short = model_short_name(d.get("llm_model", ""))
    summary[k][f"intr_scored_{short}"] = str(d.get("verdict", ""))
    summary[k][f"intr_scored_{short}_time"] = d.get("usage", {}).get("elapsed_seconds", 0)
    summary[k][f"intr_scored_{short}_tokens"] = d.get("usage", {}).get("total_tokens", 0)
    summary[k][f"intr_scored_{short}_cost"] = d.get("cost_usd", 0)

# Extrinsic: take worst across ref APs per domain
for d in extr_sim_data:
    k = (d.get("cve_id", ""), d.get("test_ap", d.get("ap_id", "")))
    model = model_short_name(d.get("model", "unknown"))
    col = f"extr_sim_{model}"
    prev = summary[k].get(col, "True")
    cur = d.get("prediction", "")
    summary[k][col] = "False" if cur == "False" or prev == "False" else cur

for d in extr_bin_data:
    k = (d.get("cve_id", ""), d.get("generated", ""))
    short = model_short_name(d.get("llm_model", ""))
    col = f"extr_bin_{short}"
    prev = summary[k].get(col, "True")
    cur = d.get("label", "")
    summary[k][col] = "False" if cur == "False" or prev == "False" else cur

for d in extr_scored_data:
    k = (d.get("cve_id", ""), d.get("generated", ""))
    short = model_short_name(d.get("llm_model", ""))
    col = f"extr_scored_{short}"
    prev_min = summary[k].get(f"{col}_min", 999)
    cur_min = d.get("min_score")
    if cur_min is not None and (prev_min == 999 or cur_min < prev_min):
        summary[k][col] = str(d.get("verdict", ""))
        summary[k][f"{col}_min"] = cur_min
        summary[k][f"{col}_cost"] = d.get("cost_usd", 0)

# ── Discover all column names dynamically ──
all_cols = set()
for v in summary.values():
    all_cols.update(v.keys())

# Separate into categories
# Order: syntax → solvability → intrinsic sim → intrinsic bin → intrinsic scored → extrinsic sim → extrinsic bin → extrinsic scored
_eval_raw = [c for c in all_cols if not c.endswith(("_time", "_tokens", "_cost", "_min")) 
             and c not in ("timestamp", "source", "cve_id", "ap_id")]

_order_prefix = [
    "syntax", "solvability",
    "intr_sim_", "intr_bin_", "intr_scored_",
    "extr_sim_", "extr_bin_", "extr_scored_",
]

def _col_sort_key(col):
    for i, prefix in enumerate(_order_prefix):
        if col == prefix.rstrip("_") or col.startswith(prefix):
            return (i, col)
    return (len(_order_prefix), col)

eval_cols = sorted(_eval_raw, key=_col_sort_key)
time_cols = sorted([c for c in all_cols if c.endswith("_time")])
token_cols = sorted([c for c in all_cols if c.endswith("_tokens")])
cost_cols = sorted([c for c in all_cols if c.endswith("_cost")])

# ── Build DataFrame ──
rows = []
for k, v in sorted(summary.items()):
    row = {
        "timestamp": v.get("timestamp", ""),
        "domain": f"{v.get('cve_id', k[0])}/{v.get('ap_id', k[1])}",
        "source": v.get("source", ""),
    }
    # Add all evaluation columns
    for c in eval_cols:
        row[c] = v.get(c, "")
    # Totals
    row["total_time_s"] = round(sum(v.get(c, 0) or 0 for c in time_cols if isinstance(v.get(c), (int, float))), 2)
    row["total_tokens"] = sum(v.get(c, 0) or 0 for c in token_cols if isinstance(v.get(c), (int, float)))
    row["total_cost_usd"] = round(sum(v.get(c, 0) or 0 for c in cost_cols if isinstance(v.get(c), (int, float))), 6)
    rows.append(row)

df = pd.DataFrame(rows)

# ── Print ──
print("=" * 120)
print("EVALUATION SUMMARY")
print("=" * 120)
print(df.to_string(index=False))

# ── Kappa ──
print("\n--- Fleiss\'  Kappa (inter-rater agreement) ---")
if kappa_data:
    kd = kappa_data[0]  # single record
    print(f"  Kappa = {kd.get('kappa', 'N/A')}  ({kd.get('interpretation', '')})")
    print(f"  Subjects: {kd.get('n_subjects', '')}, Raters: {kd.get('n_raters', '')}")
    print(f"  Raters: {kd.get('raters', [])}")
else:
    print("  No kappa data found")

# ── Totals ──
print(f"\n--- Totals ---")
print(f"  Domains evaluated: {len(df)}")
print(f"  Total time: {df['total_time_s'].sum():.2f}s")
print(f"  Total tokens: {int(df['total_tokens'].sum())}")
print(f"  Total cost: ${df['total_cost_usd'].sum():.4f}")

# ── Save ──
summary_path = os.path.join(RESULTS_BASE, "results_summary.jsonl")
with open(summary_path, "w") as f:
    for _, row in df.iterrows():
        f.write(json.dumps(row.to_dict()) + "\n")


EVALUATION SUMMARY
timestamp                                              domain    source syntax solvability intr_sim_all-MiniLM-L6-v2 intr_sim_bge-base-en-v1.5 intr_bin_gpt-4.1-mini intr_bin_llama-3.3-70b intr_scored_gpt-4.1-mini intr_scored_llama-3.3-70b extr_sim_all-MiniLM-L6-v2 extr_sim_bge-base-en-v1.5 extr_bin_gpt-4.1-mini extr_bin_llama-3.3-70b extr_scored_gpt-4.1-mini extr_scored_llama-3.3-70b  total_time_s  total_tokens  total_cost_usd
                                           CVE-2022-40149/AP2 reference   True        True                     False                     False                  True                   True                  True(5)                   True(5)                      True                      True                  True                  False                  True(5)                   True(4)         40.29         21007        0.017260
          CVE-2024-38809/AP1_inject_capability_violation_rep1       bad   True        True                      True   